In [ ]:
# Load the CSV file into an R data frame
df <- read.csv("/content/CRCALVS_8_1.csv")

# Print column names
cat("Columns in the DataFrame:\n")
print(colnames(df))

# Print random 5 rows from the data frame
cat("\nRandom 5 rows from the DataFrame:\n")
print(df[sample(nrow(df), 5), ])

Columns in the DataFrame:
 [1] "Index"           "Time_survival"   "Time_recurrence" "Outcome_OS"     
 [5] "Outcome_DSS"     "Outcome_DFS"     "X"               "AG"             
 [9] "GN"              "ST_main"         "SN_main"         "LN_total"       
[13] "LN_positive"     "LNR"             "Grade"           "LVI"            

Random 5 rows from the DataFrame:
   Index Time_survival Time_recurrence Outcome_OS Outcome_DSS Outcome_DFS  X AG
96   768          2541            2541          0           0           0 NA 60
75   744          3227            3227          0           0           0 NA 50
61   720          1085            1085          1           1           0 NA 54
29   654             4               4          1           1           0 NA 61
7    613           340             340          1           1           0 NA 33
   GN ST_main SN_main LN_total LN_positive    LNR Grade LVI
96  1       3       1       15           4  26.67     1   1
75  0       3       0       14 

## 0

In [ ]:
################################################################################
#  00_config.R — Master Configuration for LNR vs N-Stage Analysis
#  Protocol: LNR vs TNM N-Stage as a Prognostic Marker in Node-Positive CRC
#  Version 2.0 | Post-Critical-Review Edition
#
#  This script is sourced by ALL subsequent analysis scripts.
#  Modify settings here ONLY — downstream scripts inherit everything.
################################################################################

# ═══════════════════════════════════════════════════════════════════════════════
# 1. PACKAGE MANAGEMENT
# ═══════════════════════════════════════════════════════════════════════════════

# Set up user library path (system library may be read-only)
user_lib <- Sys.getenv("R_LIBS_USER")
if (!dir.exists(user_lib)) dir.create(user_lib, recursive = TRUE)
.libPaths(c(user_lib, .libPaths()))

required_packages <- c(
  # -- Core survival analysis
  "survival",       # Cox PH, Surv objects, survfit, cox.zph
  "rms",            # Restricted cubic splines, cph, validate, calibrate
  "survminer",      # Publication-quality KM curves (ggsurvplot)

  # -- Competing risks
  "cmprsk",         # Fine-Gray subdistribution hazard models
  "tidycmprsk",     # Tidy interface for cmprsk

  # -- Performance metrics
  # Note: Uno's C-statistic is computed via survival::concordance(timewt="n/G2")
  "dcurves",        # Decision-curve analysis
  "pec",            # Prediction error curves (alternative C-stat)

  # -- Multiple imputation
  "mice",           # Multiple imputation by chained equations

  # -- Tables and reporting
  "tableone",       # Baseline characteristics tables with SMD
  "knitr",          # Table formatting
  "kableExtra",     # Enhanced table output
  "gtsummary",      # Publication-ready summary tables


  # -- Graphics
  "ggplot2",        # Core plotting
  "patchwork",      # Multi-panel figure assembly
  "scales",         # Axis formatting
  "RColorBrewer",   # Colour palettes
  "viridis",        # Colourblind-safe palettes
  "ggrepel",        # Non-overlapping text labels

  # -- Utilities
  "dplyr",          # Data manipulation
  "tidyr",          # Data reshaping
  "forcats",        # Factor manipulation
  "boot",           # Bootstrap resampling
  "broom"           # Tidy model outputs
)

# Install missing packages
install_if_missing <- function(pkgs) {
  missing <- pkgs[!pkgs %in% installed.packages()[, "Package"]]
  if (length(missing) > 0) {
    cat("Installing missing packages:", paste(missing, collapse = ", "), "\n")
    install.packages(missing, repos = "https://cloud.r-project.org",
                     dependencies = TRUE, quiet = TRUE)
  }
}
install_if_missing(required_packages)

# Load all packages
invisible(lapply(required_packages, function(pkg) {
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}))

cat("All packages loaded successfully.\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. FILE PATHS
# ═══════════════════════════════════════════════════════════════════════════════

# -- Base project directory
DIR_PROJECT <- file.path("d:/Google_SSD_RAM/_OneDrive/Research",
                         "Project_ColoRectal/LNR_20260328")

# -- Analysis scripts directory (this folder)
DIR_ANALYSIS <- file.path(DIR_PROJECT, "Analysis")

# -- Input data
FILE_RAW_DATA <- file.path("/content/CRCALVS_8_1.csv")

# -- Output directories (created if they don't exist)
DIR_OUTPUT    <- file.path(DIR_ANALYSIS, "Outputs")
DIR_TABLES    <- file.path(DIR_OUTPUT, "Tables")
DIR_FIGURES   <- file.path(DIR_OUTPUT, "Figures")
DIR_DATA      <- file.path(DIR_OUTPUT, "Data")
DIR_MODELS    <- file.path(DIR_OUTPUT, "Models")
DIR_APPENDIX  <- file.path(DIR_OUTPUT, "Appendix")

for (d in c(DIR_OUTPUT, DIR_TABLES, DIR_FIGURES, DIR_DATA, DIR_MODELS, DIR_APPENDIX)) {
  if (!dir.exists(d)) dir.create(d, recursive = TRUE)
}

# ═══════════════════════════════════════════════════════════════════════════════
# 3. REPRODUCIBILITY
# ═══════════════════════════════════════════════════════════════════════════════

SEED <- 2024
set.seed(SEED)

# ═══════════════════════════════════════════════════════════════════════════════
# 4. ANALYTIC CONSTANTS (from Protocol v2.0)
# ═══════════════════════════════════════════════════════════════════════════════

# --- Node-Positive Filtering Strategy ---
# Set to "SN_main" to filter by N-stage (SN_main >= 1)
# Set to "LN_positive" to filter by positive lymph node count (LN_positive >= 1)
# The user can toggle this to compare both approaches
NODE_POSITIVE_FILTER <- "LN_positive"   # <-- ADJUSTABLE: "SN_main" or "LN_positive"

# --- LNR Rosenberg (2008) categorical cutpoints (on proportion scale 0-1) ---
LNR_CUT_LOW_INTER  <- 0.05    # Boundary between Low and Intermediate
LNR_CUT_INTER_HIGH <- 0.20    # Boundary between Intermediate and High
LNR_TIER_LABELS    <- c("Low (<0.05)", "Intermediate (0.05-0.20)", "High (>0.20)")

# --- Alternative LNR cutpoints for sensitivity analyses ---
LNR_ALT_CUT_1 <- c(0.05, 0.40)    # Sensitivity: wider High tier
LNR_ALT_CUT_2 <- 0.20             # Sensitivity: single binary cutpoint

# --- TLE (Total Lymph nodes Examined) adequacy thresholds ---
TLE_ADEQUATE_THRESHOLD <- 12       # AJCC minimum adequacy threshold
TLE_STRINGENT_THRESHOLD <- 8       # Stringent adequacy criterion

# --- RCS (Restricted Cubic Splines) specification ---
RCS_KNOTS_DEFAULT <- 3             # Number of knots for RCS (yields 2 df)
# Knots placed at 10th, 50th, 90th percentiles by default (rms::rcs default)

# --- Uno C-statistic ---
UNO_TRUNCATION_QUANTILE <- 0.75    # Truncation at 75th percentile of event times

# --- Bootstrap ---
N_BOOTSTRAP <- 1000                # Number of bootstrap replicates
BOOTSTRAP_METHOD <- "bca"          # Bias-corrected and accelerated

# --- DCA (Decision Curve Analysis) ---
DCA_THRESHOLD_MIN <- 0.10
DCA_THRESHOLD_MAX <- 0.60
DCA_THRESHOLD_STEP <- 0.01
DCA_THRESHOLDS <- seq(DCA_THRESHOLD_MIN, DCA_THRESHOLD_MAX, by = DCA_THRESHOLD_STEP)

# --- Multiple Imputation ---
MICE_M <- 20                       # Number of imputed datasets
MICE_MISSINGNESS_THRESHOLD <- 0.05 # Apply MICE only if missingness > 5%

# --- Significance and decision thresholds ---
ALPHA <- 0.05                      # Two-sided alpha
DELTA_C_THRESHOLD <- 0.05          # Minimum clinically meaningful ΔC
PH_VIOLATION_THRESHOLD <- 0.10     # Schoenfeld residual p threshold for PH

# --- Landmark analysis ---
LANDMARK_DAYS <- 180               # 6-month landmark (in days)

# --- CSS conditional inclusion ---
CSS_COD_THRESHOLD <- 0.85          # CSS included only if ≥85% cause-of-death data

# --- KM survival time points (years) ---
SURV_TIMEPOINTS_YEARS <- c(1, 3, 5)
SURV_TIMEPOINTS_DAYS  <- SURV_TIMEPOINTS_YEARS * 365.25

# --- RMST truncation ---
RMST_TAU_YEARS <- 5
RMST_TAU_DAYS  <- RMST_TAU_YEARS * 365.25

# ═══════════════════════════════════════════════════════════════════════════════
# 5. EVENT-COUNT GATING FRAMEWORK (Protocol Section 6.3)
# ═══════════════════════════════════════════════════════════════════════════════

#' Determine analytic tier based on observed death event count
#' @param n_events Integer: total death events (Outcome_OS == 1)
#' @return Named list with tier label, permitted analyses description,
#'         max_df, and boolean flags for what is permitted
get_analytic_tier <- function(n_events) {
  if (n_events < 80) {
    tier <- list(
      label        = "Inadequate (< 80 events)",
      tier_code    = 1,
      max_df       = 2,
      permit_mva   = FALSE,
      permit_rcs   = FALSE,
      permit_dca   = FALSE,
      permit_interaction = FALSE,
      permit_subgroup_inference = FALSE,
      description  = paste0(
        "KM curves + log-rank test only. Univariate Cox for LNR and N-stage. ",
        "No multivariable adjustment. Findings = hypothesis-generating."
      )
    )
  } else if (n_events < 130) {
    tier <- list(
      label        = "Minimum Viable (80-129 events)",
      tier_code    = 2,
      max_df       = 8,
      permit_mva   = TRUE,
      permit_rcs   = FALSE,   # Use linear terms, not RCS
      permit_dca   = TRUE,
      permit_interaction = FALSE,
      permit_subgroup_inference = FALSE,
      description  = paste0(
        "Restricted MVA: max 8 df, linear terms for LNR & TLE. ",
        "C-statistic + DCA + calibration. No interaction tests."
      )
    )
  } else if (n_events < 200) {
    tier <- list(
      label        = "Target (130-199 events)",
      tier_code    = 3,
      max_df       = 16,
      permit_mva   = TRUE,
      permit_rcs   = TRUE,
      permit_dca   = TRUE,
      permit_interaction = FALSE,
      permit_subgroup_inference = FALSE,
      description  = paste0(
        "Full MVA with RCS splines. Uno C-stat, DCA, calibration. ",
        "Fine-Gray for CSS. Subgroups descriptive only."
      )
    )
  } else {
    tier <- list(
      label        = "Full Power (>= 200 events)",
      tier_code    = 4,
      max_df       = 20,
      permit_mva   = TRUE,
      permit_rcs   = TRUE,
      permit_dca   = TRUE,
      permit_interaction = TRUE,
      permit_subgroup_inference = TRUE,
      description  = paste0(
        "All analyses + formal LNR x TLE interaction. ",
        "Subgroup analyses with inferential C-stat."
      )
    )
  }
  tier$n_events <- n_events
  return(tier)
}

# ═══════════════════════════════════════════════════════════════════════════════
# 6. PUBLICATION THEME AND COLOURS
# ═══════════════════════════════════════════════════════════════════════════════

# -- Colour palettes --
COL_NSTAGE <- c("N1" = "#2166AC", "N2" = "#B2182B")

COL_LNR_TIERS <- c(
  "Low (<0.05)"              = "#4DAF4A",
  "Intermediate (0.05-0.20)" = "#FF7F00",
  "High (>0.20)"             = "#E41A1C"
)

COL_MODELS <- c(
  "Model A (N-stage)"          = "#2166AC",
  "Model B (LNR)"              = "#B2182B",
  "Model C2 (N-stage + TLE)"   = "#4393C3",
  "Model D2 (LNR + TLE)"       = "#D6604D",
  "Model C (N-stage Full MVA)"  = "#053061",
  "Model D (LNR Full MVA)"      = "#67001F"
)

COL_TLE <- c("Inadequate (<12)" = "#D95F02", "Adequate (>=12)" = "#1B9E77")

# -- Publication ggplot2 theme --
theme_publication <- function(base_size = 12) {
  theme_bw(base_size = base_size) +
    theme(
      # Text
      plot.title       = element_text(face = "bold", size = base_size + 2,
                                       hjust = 0),
      plot.subtitle    = element_text(size = base_size, colour = "grey40"),
      plot.caption     = element_text(size = base_size - 2, colour = "grey50",
                                       hjust = 0),
      # Axes
      axis.title       = element_text(face = "bold", size = base_size),
      axis.text        = element_text(size = base_size - 1),
      axis.line        = element_line(colour = "black", linewidth = 0.4),
      # Legend
      legend.title     = element_text(face = "bold", size = base_size - 1),
      legend.text      = element_text(size = base_size - 2),
      legend.background = element_rect(fill = "white", colour = NA),
      legend.key       = element_rect(fill = "white", colour = NA),
      legend.position  = "bottom",
      # Panel
      panel.border     = element_blank(),
      panel.grid.major = element_line(colour = "grey90", linewidth = 0.3),
      panel.grid.minor = element_blank(),
      # Strip (facets)
      strip.background = element_rect(fill = "grey95", colour = NA),
      strip.text       = element_text(face = "bold", size = base_size - 1),
      # Margins
      plot.margin      = margin(10, 10, 10, 10)
    )
}

# Set as default theme
theme_set(theme_publication())

# -- Figure export settings --
FIG_WIDTH  <- 8    # inches
FIG_HEIGHT <- 6    # inches
FIG_DPI    <- 300  # dots per inch

# ═══════════════════════════════════════════════════════════════════════════════
# 7. HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

#' Save a ggplot figure to both PDF and PNG
#' @param plot ggplot object
#' @param filename Base filename without extension
#' @param width Width in inches (default FIG_WIDTH)
#' @param height Height in inches (default FIG_HEIGHT)
save_figure <- function(plot, filename, width = FIG_WIDTH, height = FIG_HEIGHT,
                        subdir = DIR_FIGURES) {
  ggsave(file.path(subdir, paste0(filename, ".pdf")),
         plot = plot, width = width, height = height, dpi = FIG_DPI)
  ggsave(file.path(subdir, paste0(filename, ".png")),
         plot = plot, width = width, height = height, dpi = FIG_DPI)
  cat("Saved:", filename, "(PDF + PNG)\n")
}

#' Save a table as CSV and return formatted kable
#' @param df Data frame
#' @param filename Base filename without extension
save_table <- function(df, filename, subdir = DIR_TABLES) {
  write.csv(df, file.path(subdir, paste0(filename, ".csv")),
            row.names = FALSE)
  cat("Saved:", filename, ".csv\n")
}

#' Print a section header to console
section_header <- function(title) {
  width <- 78
  cat("\n", strrep("=", width), "\n", sep = "")
  cat("  ", title, "\n", sep = "")
  cat(strrep("=", width), "\n\n", sep = "")
}

#' Format p-value for display
format_pval <- function(p, digits = 3) {
  ifelse(p < 0.001, "< 0.001",
         ifelse(p < 0.01, sprintf("%.3f", p),
                sprintf(paste0("%.", digits, "f"), p)))
}

#' Wilson confidence interval for a proportion
wilson_ci <- function(x, n, alpha = ALPHA) {
  z <- qnorm(1 - alpha / 2)
  p_hat <- x / n
  denom <- 1 + z^2 / n
  centre <- (p_hat + z^2 / (2 * n)) / denom
  margin <- (z / denom) * sqrt(p_hat * (1 - p_hat) / n + z^2 / (4 * n^2))
  c(lower = max(0, centre - margin),
    estimate = p_hat,
    upper = min(1, centre + margin))
}

cat("\n", strrep("=", 78), "\n", sep = "")
cat("  CONFIG LOADED SUCCESSFULLY\n")
cat("  Node-positive filter: ", NODE_POSITIVE_FILTER, "\n")
cat("  LNR cutpoints (Rosenberg): ", LNR_CUT_LOW_INTER, "/", LNR_CUT_INTER_HIGH, "\n")
cat("  TLE adequacy threshold: ", TLE_ADEQUATE_THRESHOLD, "\n")
cat("  Bootstrap replicates: ", N_BOOTSTRAP, "\n")
cat("  Random seed: ", SEED, "\n")
cat(strrep("=", 78), "\n")


Installing missing packages: rms, survminer, cmprsk, tidycmprsk, dcurves, pec, mice, tableone, kableExtra, gtsummary, patchwork, viridis, ggrepel 


also installing the dependencies ‘fracdiff’, ‘lmtest’, ‘urca’, ‘tensorA’, ‘distributional’, ‘bdsmatrix’, ‘ucminf’, ‘AsioHeaders’, ‘Deriv’, ‘forecast’, ‘microbenchmark’, ‘fontBitstreamVera’, ‘fontLiberation’, ‘matrixStats’, ‘posterior’, ‘corrplot’, ‘bbmle’, ‘fastGHQuad’, ‘lsoda’, ‘libcoin’, ‘ordinal’, ‘rbibutils’, ‘gtools’, ‘websocket’, ‘V8’, ‘reactR’, ‘doBy’, ‘fontquiver’, ‘clock’, ‘gower’, ‘ipred’, ‘timeDate’, ‘wk’, ‘Formula’, ‘MatrixModels’, ‘mvtnorm’, ‘TH.data’, ‘sandwich’, ‘checkmate’, ‘lazyeval’, ‘crosstalk’, ‘coda’, ‘MLEcens’, ‘RcppEigen’, ‘loo’, ‘BH’, ‘RcppParallel’, ‘StanHeaders’, ‘ggsci’, ‘cowplot’, ‘ggsignif’, ‘polynom’, ‘rstatix’, ‘exactRankTests’, ‘gridtext’, ‘assertthat’, ‘deSolve’, ‘mstate’, ‘muhaz’, ‘numDeriv’, ‘quadprog’, ‘rstpm2’, ‘statmod’, ‘litedown’, ‘sparsevctrs’, ‘rex’, ‘hunspell’, ‘diagram’, ‘iterators’, ‘doParallel’, ‘mets’, ‘plotrix’, ‘Publish’, ‘RcppArmadillo’, ‘future.apply’, ‘progressr’, ‘SQUAREM’, ‘modeltools’, ‘strucchange’, ‘coin’, ‘shape’, ‘jomo’, ‘globa

All packages loaded successfully.

  CONFIG LOADED SUCCESSFULLY
  Node-positive filter:  LN_positive 
  LNR cutpoints (Rosenberg):  0.05 / 0.2 
  TLE adequacy threshold:  12 
  Bootstrap replicates:  1000 
  Random seed:  2024 


In [ ]:
################################################################################
#  01_data_preparation.R — Data Loading, Cleaning, and Cohort Selection
#  Protocol: LNR vs TNM N-Stage Prognostic Comparison in Node-Positive CRC
#
#  Inputs:  Raw CSV (CRCALVS_8_1.csv)
#  Outputs: Cleaned analytic dataset (.RData + .csv), CONSORT flow, gating tier
#
#  Source 00_config.R before running this script.
################################################################################

section_header("SCRIPT 01: DATA PREPARATION")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD RAW DATA
# ═══════════════════════════════════════════════════════════════════════════════

cat("Loading raw data from:", FILE_RAW_DATA, "\n")
df_raw <- read.csv(FILE_RAW_DATA, header = TRUE, stringsAsFactors = FALSE)

cat("  Raw dataset dimensions:", nrow(df_raw), "rows x", ncol(df_raw), "columns\n")

# Drop useless column X
if ("X" %in% names(df_raw)) {
  df_raw$X <- NULL
  cat("  Dropped column 'X' (empty/useless).\n")
}

# Remove any completely empty trailing rows
df_raw <- df_raw[!is.na(df_raw$Index) & df_raw$Index != "", ]
cat("  After removing empty rows:", nrow(df_raw), "patients\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. DATA INTEGRITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("DATA INTEGRITY CHECKS")

# --- 2.1 Check for duplicate Index values ---
n_dup <- sum(duplicated(df_raw$Index))
cat("Duplicate patient IDs:", n_dup, "\n")
if (n_dup > 0) {
  cat("  WARNING: Duplicates found at Index:",
      df_raw$Index[duplicated(df_raw$Index)], "\n")
}

# --- 2.2 Check LN_positive <= LN_total ---
invalid_ln <- df_raw$LN_positive > df_raw$LN_total
n_invalid_ln <- sum(invalid_ln, na.rm = TRUE)
cat("Records where LN_positive > LN_total:", n_invalid_ln, "\n")
if (n_invalid_ln > 0) {
  cat("  *** DATA ERROR: The following records have positive > total nodes:\n")
  print(df_raw[which(invalid_ln), c("Index", "LN_total", "LN_positive", "LNR")])
  cat("  These records will be EXCLUDED per protocol (Section 4.4).\n")
}

# --- 2.3 Validate LNR against recomputed value ---
# Dataset LNR is stored as percentage (0-100)
df_raw$LNR_recomputed <- ifelse(df_raw$LN_total > 0,
                                 (df_raw$LN_positive / df_raw$LN_total) * 100,
                                 0)
lnr_discrepancy <- abs(df_raw$LNR - df_raw$LNR_recomputed) > 0.5
n_discrepancy <- sum(lnr_discrepancy, na.rm = TRUE)
cat("LNR discrepancies (|stored - recomputed| > 0.5%):", n_discrepancy, "\n")
if (n_discrepancy > 0) {
  cat("  Records with discrepancies:\n")
  print(df_raw[which(lnr_discrepancy),
               c("Index", "LN_total", "LN_positive", "LNR", "LNR_recomputed")])
}

# --- 2.4 Check value ranges ---
cat("\nValue range checks:\n")
cat("  Time_survival: [", min(df_raw$Time_survival, na.rm = TRUE), ",",
    max(df_raw$Time_survival, na.rm = TRUE), "] days\n")
cat("  Age (AG): [", min(df_raw$AG, na.rm = TRUE), ",",
    max(df_raw$AG, na.rm = TRUE), "] years\n")
cat("  T-stage (ST_main): ", paste(sort(unique(df_raw$ST_main)), collapse = ", "), "\n")
cat("  N-stage (SN_main): ", paste(sort(unique(df_raw$SN_main)), collapse = ", "), "\n")
cat("  LN_total: [", min(df_raw$LN_total, na.rm = TRUE), ",",
    max(df_raw$LN_total, na.rm = TRUE), "]\n")
cat("  LN_positive: [", min(df_raw$LN_positive, na.rm = TRUE), ",",
    max(df_raw$LN_positive, na.rm = TRUE), "]\n")
cat("  Grade: ", paste(sort(unique(df_raw$Grade)), collapse = ", "), "\n")
cat("  LVI: ", paste(sort(unique(df_raw$LVI)), collapse = ", "), "\n")
cat("  Outcome_OS: ", paste(sort(unique(df_raw$Outcome_OS)), collapse = ", "), "\n")
cat("  Outcome_DSS: ", paste(sort(unique(df_raw$Outcome_DSS)), collapse = ", "), "\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. MISSINGNESS ASSESSMENT
# ═══════════════════════════════════════════════════════════════════════════════

section_header("MISSINGNESS ASSESSMENT")

analysis_vars <- c("Time_survival", "Outcome_OS", "Outcome_DSS",
                    "AG", "GN", "ST_main", "SN_main",
                    "LN_total", "LN_positive", "LNR", "Grade", "LVI")

missing_summary <- data.frame(
  Variable = analysis_vars,
  N_missing = sapply(analysis_vars, function(v) sum(is.na(df_raw[[v]]))),
  Pct_missing = sapply(analysis_vars, function(v)
    round(100 * sum(is.na(df_raw[[v]])) / nrow(df_raw), 2))
)
print(missing_summary)

any_high_missing <- any(missing_summary$Pct_missing > MICE_MISSINGNESS_THRESHOLD * 100)
cat("\nAny variable with >", MICE_MISSINGNESS_THRESHOLD * 100, "% missingness:",
    any_high_missing, "\n")
if (any_high_missing) {
  cat("  --> MICE multiple imputation will be applied in downstream scripts.\n")
} else {
  cat("  --> Complete case analysis is the primary approach (no imputation needed).\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 4. CONSORT-STYLE PATIENT FLOW (Tracking all exclusions)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("CONSORT-STYLE PATIENT FLOW")

flow <- list()
flow$step1_total <- nrow(df_raw)
cat("Step 1 — Total records in database:", flow$step1_total, "\n")

# --- 4.1 Exclude records with invalid LN data ---
df_work <- df_raw[!invalid_ln | is.na(invalid_ln), ]
flow$excluded_invalid_ln <- flow$step1_total - nrow(df_work)
flow$step2_after_ln_valid <- nrow(df_work)
cat("Step 2 — After excluding invalid LN (positive > total):",
    flow$step2_after_ln_valid,
    "(excluded:", flow$excluded_invalid_ln, ")\n")

# --- 4.2 Exclude records with missing survival data ---
has_surv <- !is.na(df_work$Time_survival) & !is.na(df_work$Outcome_OS)
df_work <- df_work[has_surv, ]
flow$excluded_missing_surv <- flow$step2_after_ln_valid - nrow(df_work)
flow$step3_after_surv <- nrow(df_work)
cat("Step 3 — After excluding missing survival data:",
    flow$step3_after_surv,
    "(excluded:", flow$excluded_missing_surv, ")\n")

# --- 4.3 Store the FULL dataset (node-positive + node-negative) ---
df_all <- df_work
cat("\n  Full cohort (all nodes) retained for N-stage-only models: N =",
    nrow(df_all), "\n")

# --- 4.4 Filter to NODE-POSITIVE cohort ---
cat("\n  >>> Filtering strategy: ", NODE_POSITIVE_FILTER, " <<<\n")

if (NODE_POSITIVE_FILTER == "SN_main") {
  node_pos_mask <- df_work$SN_main >= 1
  filter_description <- "SN_main >= 1 (N-stage N1 or N2)"
} else if (NODE_POSITIVE_FILTER == "LN_positive") {
  node_pos_mask <- df_work$LN_positive >= 1
  filter_description <- "LN_positive >= 1 (at least 1 positive lymph node)"
} else {
  stop("Invalid NODE_POSITIVE_FILTER value. Must be 'SN_main' or 'LN_positive'.")
}

df_nodepos <- df_work[node_pos_mask, ]
flow$excluded_node_negative <- nrow(df_work) - nrow(df_nodepos)
flow$step4_node_positive <- nrow(df_nodepos)
cat("Step 4 — Node-positive cohort (", filter_description, "):",
    flow$step4_node_positive,
    "(excluded node-negative:", flow$excluded_node_negative, ")\n")

# --- 4.5 Exclude patients with missing TLE (cannot compute LNR) ---
tle_missing_mask <- is.na(df_nodepos$LN_total) | df_nodepos$LN_total == 0
n_tle_missing <- sum(tle_missing_mask)
if (n_tle_missing > 0) {
  cat("  NOTE:", n_tle_missing, "node-positive patients have missing/zero TLE. ",
      "They are excluded from LNR models but retained for N-stage-only.\n")
  df_nodepos_tle_avail <- df_nodepos[!tle_missing_mask, ]
} else {
  df_nodepos_tle_avail <- df_nodepos
}
flow$excluded_tle_missing <- n_tle_missing
flow$step5_final_analytic <- nrow(df_nodepos_tle_avail)
cat("Step 5 — Final analytic cohort (LNR computable):",
    flow$step5_final_analytic, "\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. CREATE DERIVED VARIABLES
# ═══════════════════════════════════════════════════════════════════════════════

section_header("CREATING DERIVED VARIABLES")

df <- df_nodepos_tle_avail  # Working analytic dataset

# --- 5.1 LNR as proportion (0-1) — recomputed from raw counts ---
df$LNR_prop <- df$LN_positive / df$LN_total
cat("LNR_prop (proportion 0-1) created. Range: [",
    round(min(df$LNR_prop, na.rm = TRUE), 4), ",",
    round(max(df$LNR_prop, na.rm = TRUE), 4), "]\n")

# --- 5.2 LNR categorical tiers (Rosenberg 2008) ---
df$LNR_tier <- cut(df$LNR_prop,
                    breaks = c(-Inf, LNR_CUT_LOW_INTER, LNR_CUT_INTER_HIGH, Inf),
                    labels = LNR_TIER_LABELS,
                    right = FALSE)
cat("LNR tier distribution:\n")
print(table(df$LNR_tier, useNA = "ifany"))

# --- 5.3 Alternative LNR categorisations for sensitivity ---
df$LNR_tier_alt1 <- cut(df$LNR_prop,
                         breaks = c(-Inf, LNR_ALT_CUT_1, Inf),
                         labels = c("Low (<0.05)", "Intermediate (0.05-0.40)",
                                    "High (>0.40)"),
                         right = FALSE)

df$LNR_tier_binary <- ifelse(df$LNR_prop < LNR_ALT_CUT_2,
                              "Low (<0.20)", "High (>=0.20)")
df$LNR_tier_binary <- factor(df$LNR_tier_binary,
                              levels = c("Low (<0.20)", "High (>=0.20)"))

# --- 5.4 N-stage factor ---
df$N_stage <- factor(df$SN_main, levels = c(1, 2), labels = c("N1", "N2"))
cat("\nN-stage distribution:\n")
print(table(df$N_stage, useNA = "ifany"))

# --- 5.5 T-stage grouped (per protocol: T1/T2 vs T3 vs T4) ---
df$T_stage_group <- factor(
  ifelse(df$ST_main <= 2, "T1/T2",
         ifelse(df$ST_main == 3, "T3", "T4")),
  levels = c("T1/T2", "T3", "T4")
)
cat("\nT-stage group distribution:\n")
print(table(df$T_stage_group, useNA = "ifany"))

# --- 5.6 TLE adequacy ---
df$TLE_adequate <- ifelse(df$LN_total >= TLE_ADEQUATE_THRESHOLD,
                           "Adequate (>=12)", "Inadequate (<12)")
df$TLE_adequate <- factor(df$TLE_adequate,
                           levels = c("Inadequate (<12)", "Adequate (>=12)"))
df$TLE_adequate_binary <- as.integer(df$LN_total >= TLE_ADEQUATE_THRESHOLD)
cat("\nTLE adequacy distribution:\n")
print(table(df$TLE_adequate, useNA = "ifany"))

# Stringent threshold
df$TLE_stringent <- ifelse(df$LN_total >= TLE_STRINGENT_THRESHOLD,
                            "Adequate (>=8)", "Inadequate (<8)")
df$TLE_stringent <- factor(df$TLE_stringent,
                            levels = c("Inadequate (<8)", "Adequate (>=8)"))

# --- 5.7 Grade factor ---
df$Grade_f <- factor(df$Grade, levels = c(0, 1),
                      labels = c("Low grade", "High grade"))
if (all(df$Grade %in% c(1, 2), na.rm = TRUE)) {
  # Re-check: data shows Grade = 1 (low/high? 2=high grade)
  # Based on user: 0=low grade, 1=high grade
  # But data shows 1 and 2 — need to re-map
  df$Grade_f <- factor(df$Grade, levels = c(1, 2),
                        labels = c("Low grade", "High grade"))
}
cat("\nGrade distribution:\n")
print(table(df$Grade_f, useNA = "ifany"))

# --- 5.8 LVI factor ---
df$LVI_f <- factor(df$LVI, levels = c(0, 1),
                    labels = c("Absent", "Present"))
cat("\nLVI distribution:\n")
print(table(df$LVI_f, useNA = "ifany"))

# --- 5.9 Sex factor ---
df$Sex_f <- factor(df$GN, levels = c(0, 1), labels = c("Male", "Female"))
cat("\nSex distribution:\n")
print(table(df$Sex_f, useNA = "ifany"))

# --- 5.10 Survival time in months (for display) ---
df$Time_survival_months <- df$Time_survival / 30.44
df$Time_survival_years <- df$Time_survival / 365.25

# ═══════════════════════════════════════════════════════════════════════════════
# 6. EVENT-COUNT GATING
# ═══════════════════════════════════════════════════════════════════════════════

section_header("EVENT-COUNT GATING")

n_os_events <- sum(df$Outcome_OS == 1, na.rm = TRUE)
n_dss_events <- sum(df$Outcome_DSS == 1, na.rm = TRUE)

cat("Node-positive analytic cohort: N =", nrow(df), "\n")
cat("Overall Survival events (deaths):", n_os_events, "\n")
cat("Disease-Specific Survival events:", n_dss_events, "\n")
cat("Median follow-up (days):", round(median(df$Time_survival), 0), "\n")
cat("Median follow-up (years):", round(median(df$Time_survival_years), 2), "\n\n")

ANALYTIC_TIER <- get_analytic_tier(n_os_events)
cat("═══ ASSIGNED ANALYTIC TIER ═══\n")
cat("  Tier:", ANALYTIC_TIER$label, "\n")
cat("  Events:", ANALYTIC_TIER$n_events, "\n")
cat("  Max degrees of freedom:", ANALYTIC_TIER$max_df, "\n")
cat("  Permitted: MVA =", ANALYTIC_TIER$permit_mva,
    "| RCS =", ANALYTIC_TIER$permit_rcs,
    "| DCA =", ANALYTIC_TIER$permit_dca,
    "| Interaction =", ANALYTIC_TIER$permit_interaction, "\n")
cat("  Description:", ANALYTIC_TIER$description, "\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. SUMMARY OF ANALYTIC DATASET
# ═══════════════════════════════════════════════════════════════════════════════

section_header("ANALYTIC DATASET SUMMARY")

cat("Final analytic dataset: N =", nrow(df), "node-positive patients\n\n")

cat("Variable listing:\n")
str(df[, c("Index", "Time_survival", "Outcome_OS", "Outcome_DSS",
           "AG", "Sex_f", "T_stage_group", "N_stage",
           "LN_total", "LN_positive", "LNR_prop", "LNR_tier",
           "Grade_f", "LVI_f", "TLE_adequate")])

# ═══════════════════════════════════════════════════════════════════════════════
# 8. IDENTIFY DISCORDANCES BETWEEN FILTER METHODS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("FILTER METHOD COMPARISON")

# Show patients where SN_main and LN_positive disagree on node-positivity
discordant <- df_work[(df_work$SN_main >= 1 & df_work$LN_positive == 0) |
                       (df_work$SN_main == 0 & df_work$LN_positive >= 1), ]

cat("Patients with discordance between SN_main and LN_positive:\n")
if (nrow(discordant) > 0) {
  print(discordant[, c("Index", "SN_main", "LN_positive", "LN_total", "LNR")])
  cat("\n  Total discordant patients:", nrow(discordant), "\n")
  cat("  These patients are classified differently depending on the filter ",
      "strategy chosen in 00_config.R (NODE_POSITIVE_FILTER = '",
      NODE_POSITIVE_FILTER, "').\n", sep = "")
} else {
  cat("  No discordances found. Both filter methods yield identical cohorts.\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 9. SAVE OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SAVING OUTPUTS")

# --- Analytic dataset ---
save(df, df_all, df_raw, flow, ANALYTIC_TIER,
     file = file.path(DIR_DATA, "analytic_dataset.RData"))
cat("Saved: analytic_dataset.RData\n")

write.csv(df, file.path(DIR_DATA, "analytic_dataset_nodepositive.csv"),
          row.names = FALSE)
cat("Saved: analytic_dataset_nodepositive.csv\n")

# --- CONSORT flow summary ---
flow_df <- data.frame(
  Step = c("1. Total records in database",
           "2. Excluded: invalid LN (positive > total)",
           "3. Excluded: missing survival data",
           "4. Excluded: node-negative patients",
           "5. Excluded: missing/zero TLE",
           "Final analytic cohort"),
  N_remaining = c(flow$step1_total,
                  flow$step2_after_ln_valid,
                  flow$step3_after_surv,
                  flow$step4_node_positive,
                  flow$step5_final_analytic,
                  flow$step5_final_analytic),
  N_excluded = c(NA,
                 flow$excluded_invalid_ln,
                 flow$excluded_missing_surv,
                 flow$excluded_node_negative,
                 flow$excluded_tle_missing,
                 NA)
)
save_table(flow_df, "consort_flow")
print(flow_df)

# --- Missingness summary ---
save_table(missing_summary, "missingness_summary")

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 01 COMPLETE\n")
cat("  Analytic cohort: N =", nrow(df), "| OS events:", n_os_events,
    "| Tier:", ANALYTIC_TIER$label, "\n")
cat("  Filter method:", NODE_POSITIVE_FILTER, "\n")
cat(strrep("=", 78), "\n")



  SCRIPT 01: DATA PREPARATION

Loading raw data from: /content/CRCALVS_8_1.csv 
  Raw dataset dimensions: 137 rows x 16 columns
  Dropped column 'X' (empty/useless).
  After removing empty rows: 137 patients

  DATA INTEGRITY CHECKS

Duplicate patient IDs: 0 
Records where LN_positive > LN_total: 0 
LNR discrepancies (|stored - recomputed| > 0.5%): 0 

Value range checks:
  Time_survival: [ 3 , 4136 ] days
  Age (AG): [ 21 , 85 ] years
  T-stage (ST_main):  1, 2, 3, 4 
  N-stage (SN_main):  0, 1, 2 
  LN_total: [ 1 , 74 ]
  LN_positive: [ 0 , 18 ]
  Grade:  1, 2 
  LVI:  0, 1 
  Outcome_OS:  0, 1 
  Outcome_DSS:  0, 1 

  MISSINGNESS ASSESSMENT

                   Variable N_missing Pct_missing
Time_survival Time_survival         0           0
Outcome_OS       Outcome_OS         0           0
Outcome_DSS     Outcome_DSS         0           0
AG                       AG         0           0
GN                       GN         0           0
ST_main             ST_main         0        

## 2

In [ ]:
################################################################################
#  02_descriptive.R — Phase 1: Descriptive Statistics & TLE Characterisation
#  Protocol Section 7.1 | SO6
#
#  Outputs: Table 1, TLE distribution, LNR distribution, outcome summary
################################################################################


# Load analytic dataset
load(file.path(DIR_DATA, "analytic_dataset.RData"))

section_header("SCRIPT 02: DESCRIPTIVE STATISTICS (Phase 1)")

cat("Analytic cohort: N =", nrow(df), "node-positive patients\n")
cat("Analytic tier:", ANALYTIC_TIER$label, "\n\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. TABLE 1 — BASELINE CHARACTERISTICS
#    Stratified by: (a) LNR tier, (b) N-stage
#    Protocol: SMD instead of p-values for baseline balance assessment
# ═══════════════════════════════════════════════════════════════════════════════

section_header("TABLE 1: BASELINE CHARACTERISTICS")

table1_vars <- c("AG", "Sex_f", "T_stage_group", "LN_total", "LN_positive",
                 "LNR_prop", "Grade_f", "LVI_f", "TLE_adequate",
                 "Time_survival", "Outcome_OS", "Outcome_DSS")

cat_vars <- c("Sex_f", "T_stage_group", "Grade_f", "LVI_f", "TLE_adequate",
              "Outcome_OS", "Outcome_DSS")

nonnormal_vars <- c("Time_survival", "LN_total", "LN_positive", "LNR_prop")

# --- Table 1a: By LNR Tier ---
cat("--- Table 1a: Stratified by LNR Tier ---\n")
tab1_lnr <- CreateTableOne(
  vars = table1_vars,
  strata = "LNR_tier",
  data = df,
  factorVars = cat_vars,
  addOverall = TRUE
)
tab1_lnr_print <- print(tab1_lnr,
                          nonnormal = nonnormal_vars,
                          smd = TRUE,        # SMD instead of p-values
                          test = FALSE,       # Suppress p-values per protocol
                          printToggle = TRUE,
                          showAllLevels = TRUE)

# --- Table 1b: By N-Stage ---
cat("\n--- Table 1b: Stratified by N-Stage ---\n")
tab1_nstage <- CreateTableOne(
  vars = table1_vars,
  strata = "N_stage",
  data = df,
  factorVars = cat_vars,
  addOverall = TRUE
)
tab1_nstage_print <- print(tab1_nstage,
                            nonnormal = nonnormal_vars,
                            smd = TRUE,
                            test = FALSE,
                            printToggle = TRUE,
                            showAllLevels = TRUE)

# Save tables
save_table(as.data.frame(tab1_lnr_print), "Table1_by_LNR_tier")
save_table(as.data.frame(tab1_nstage_print), "Table1_by_N_stage")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. TLE CHARACTERISATION (SO6 — South Asian contextual contribution)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("TLE CHARACTERISATION")

# --- 2.1 Distribution summary ---
cat("TLE Distribution (node-positive cohort):\n")
cat("  N:", sum(!is.na(df$LN_total)), "\n")
cat("  Mean (SD):", round(mean(df$LN_total, na.rm = TRUE), 1),
    "(", round(sd(df$LN_total, na.rm = TRUE), 1), ")\n")
cat("  Median (IQR):", median(df$LN_total, na.rm = TRUE),
    "(", quantile(df$LN_total, 0.25, na.rm = TRUE), "-",
    quantile(df$LN_total, 0.75, na.rm = TRUE), ")\n")
cat("  Range:", min(df$LN_total, na.rm = TRUE), "-",
    max(df$LN_total, na.rm = TRUE), "\n\n")

# --- 2.2 Proportion with inadequate harvest ---
n_below_12 <- sum(df$LN_total < TLE_ADEQUATE_THRESHOLD, na.rm = TRUE)
n_below_8 <- sum(df$LN_total < TLE_STRINGENT_THRESHOLD, na.rm = TRUE)
n_tle <- sum(!is.na(df$LN_total))

ci_12 <- wilson_ci(n_below_12, n_tle)
ci_8 <- wilson_ci(n_below_8, n_tle)

cat("Harvest adequacy:\n")
cat("  TLE < 12 (AJCC minimum):  ", n_below_12, "/", n_tle,
    " (", round(ci_12["estimate"] * 100, 1), "%; 95% CI: ",
    round(ci_12["lower"] * 100, 1), "-", round(ci_12["upper"] * 100, 1), "%)\n", sep = "")
cat("  TLE < 8 (stringent):      ", n_below_8, "/", n_tle,
    " (", round(ci_8["estimate"] * 100, 1), "%; 95% CI: ",
    round(ci_8["lower"] * 100, 1), "-", round(ci_8["upper"] * 100, 1), "%)\n", sep = "")

# --- 2.3 TLE histogram ---
p_tle_hist <- ggplot(df, aes(x = LN_total)) +
  geom_histogram(binwidth = 2, fill = "#4393C3", colour = "white", alpha = 0.85) +
  geom_vline(xintercept = TLE_ADEQUATE_THRESHOLD, linetype = "dashed",
             colour = "#D95F02", linewidth = 0.8) +
  annotate("text", x = TLE_ADEQUATE_THRESHOLD + 1,
           y = Inf, vjust = 2, hjust = 0,
           label = paste0("AJCC threshold = ", TLE_ADEQUATE_THRESHOLD),
           colour = "#D95F02", fontface = "italic", size = 3.5) +
  labs(title = "Distribution of Total Lymph Nodes Examined (TLE)",
       subtitle = "Node-positive CRC cohort",
       x = "Total Lymph Nodes Examined",
       y = "Number of Patients") +
  theme_publication()
save_figure(p_tle_hist, "Fig_TLE_histogram")

# --- 2.4 TLE by N-stage ---
p_tle_nstage <- ggplot(df, aes(x = N_stage, y = LN_total, fill = N_stage)) +
  geom_boxplot(alpha = 0.7, outlier.shape = 21) +
  geom_jitter(width = 0.15, alpha = 0.4, size = 1.5) +
  geom_hline(yintercept = TLE_ADEQUATE_THRESHOLD, linetype = "dashed",
             colour = "#D95F02", linewidth = 0.5) +
  scale_fill_manual(values = COL_NSTAGE) +
  labs(title = "Total Lymph Nodes Examined by N-Stage",
       x = "N-Stage", y = "Total Lymph Nodes Examined") +
  guides(fill = "none") +
  theme_publication()
save_figure(p_tle_nstage, "Fig_TLE_by_Nstage")

# --- 2.5 TLE by LNR tier ---
p_tle_lnr <- ggplot(df, aes(x = LNR_tier, y = LN_total, fill = LNR_tier)) +
  geom_boxplot(alpha = 0.7, outlier.shape = 21) +
  geom_jitter(width = 0.15, alpha = 0.4, size = 1.5) +
  geom_hline(yintercept = TLE_ADEQUATE_THRESHOLD, linetype = "dashed",
             colour = "#D95F02", linewidth = 0.5) +
  scale_fill_manual(values = COL_LNR_TIERS) +
  labs(title = "Total Lymph Nodes Examined by LNR Tier",
       x = "LNR Tier (Rosenberg)", y = "Total Lymph Nodes Examined") +
  guides(fill = "none") +
  theme_publication()
save_figure(p_tle_lnr, "Fig_TLE_by_LNR_tier")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. LNR DISTRIBUTION
# ═══════════════════════════════════════════════════════════════════════════════

section_header("LNR DISTRIBUTION")

cat("LNR (proportion) summary:\n")
cat("  Mean (SD):", round(mean(df$LNR_prop, na.rm = TRUE), 4),
    "(", round(sd(df$LNR_prop, na.rm = TRUE), 4), ")\n")
cat("  Median (IQR):", round(median(df$LNR_prop, na.rm = TRUE), 4),
    "(", round(quantile(df$LNR_prop, 0.25, na.rm = TRUE), 4), "-",
    round(quantile(df$LNR_prop, 0.75, na.rm = TRUE), 4), ")\n")
cat("  Range:", round(min(df$LNR_prop, na.rm = TRUE), 4), "-",
    round(max(df$LNR_prop, na.rm = TRUE), 4), "\n\n")

cat("LNR Tier Distribution (Rosenberg 2008):\n")
tier_tab <- as.data.frame(table(df$LNR_tier))
names(tier_tab) <- c("LNR_Tier", "N")
tier_tab$Pct <- round(100 * tier_tab$N / sum(tier_tab$N), 1)
print(tier_tab)
save_table(tier_tab, "LNR_tier_distribution")

# --- LNR histogram ---
p_lnr_hist <- ggplot(df, aes(x = LNR_prop)) +
  geom_histogram(binwidth = 0.05, fill = "#B2182B", colour = "white", alpha = 0.85) +
  geom_vline(xintercept = LNR_CUT_LOW_INTER, linetype = "dashed",
             colour = "#4DAF4A", linewidth = 0.8) +
  geom_vline(xintercept = LNR_CUT_INTER_HIGH, linetype = "dashed",
             colour = "#FF7F00", linewidth = 0.8) +
  annotate("text", x = LNR_CUT_LOW_INTER + 0.01, y = Inf, vjust = 2, hjust = 0,
           label = "0.05", colour = "#4DAF4A", fontface = "italic", size = 3.5) +
  annotate("text", x = LNR_CUT_INTER_HIGH + 0.01, y = Inf, vjust = 2, hjust = 0,
           label = "0.20", colour = "#FF7F00", fontface = "italic", size = 3.5) +
  labs(title = "Distribution of Lymph Node Ratio (LNR)",
       subtitle = "Node-positive CRC cohort | Rosenberg thresholds shown",
       x = "LNR (proportion)",
       y = "Number of Patients") +
  scale_x_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.1)) +
  theme_publication()
save_figure(p_lnr_hist, "Fig_LNR_histogram")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. OUTCOME SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

section_header("OUTCOME SUMMARY")

n_total <- nrow(df)
n_os <- sum(df$Outcome_OS == 1, na.rm = TRUE)
n_dss <- sum(df$Outcome_DSS == 1, na.rm = TRUE)

cat("Overall Survival:\n")
cat("  Total patients:", n_total, "\n")
cat("  Deaths (any cause):", n_os, "(", round(100 * n_os / n_total, 1), "%)\n")
cat("  Censored:", n_total - n_os, "(", round(100 * (n_total - n_os) / n_total, 1), "%)\n\n")

cat("Disease-Specific Survival:\n")
cat("  Cancer deaths:", n_dss, "(", round(100 * n_dss / n_total, 1), "%)\n")
cat("  Non-cancer deaths:", n_os - n_dss, "\n")

# -- CSS data completeness check --
if (n_os > 0) {
  css_completeness <- n_dss / n_os  # Proportion of deaths with cancer-specific attribution
  # Note: this is an approximation — assumes DSS == 1 implies cancer death
  cat("\n  CSS data completeness proxy:", round(css_completeness * 100, 1), "%\n")
  cat("  (Proportion of deceased patients with DSS event recorded)\n")
  if (css_completeness >= CSS_COD_THRESHOLD) {
    cat("  --> CSS analysis PERMITTED (>= 85% threshold)\n")
  } else {
    cat("  --> CSS analysis may be limited (< 85% threshold)\n")
  }
}

# -- Median follow-up (reverse KM method) --
surv_fu <- survfit(Surv(Time_survival, 1 - Outcome_OS) ~ 1, data = df)
median_fu <- summary(surv_fu)$table["median"]
cat("\nMedian follow-up (reverse KM method):",
    round(median_fu, 0), "days (",
    round(median_fu / 365.25, 1), "years)\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. CROSS-TABULATION: N-STAGE × LNR TIER (Preview for reclassification)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("N-STAGE × LNR TIER CROSS-TABULATION")

cross_tab <- table(df$N_stage, df$LNR_tier)
cat("Counts:\n")
print(cross_tab)
cat("\nRow percentages (within each N-stage):\n")
print(round(prop.table(cross_tab, margin = 1) * 100, 1))

save_table(as.data.frame.matrix(cross_tab), "Nstage_LNRtier_crosstab")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. PATIENT FLOW DIAGRAM (Formatted Text)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("CONSORT-STYLE PATIENT FLOW")

cat("┌─────────────────────────────────────────────────┐\n")
cat("│  Total records in database: ", sprintf("%-20d", flow$step1_total), "│\n")
cat("└──────────────────────┬──────────────────────────┘\n")
cat("                       │\n")
cat("  Excluded: invalid LN │ n =", flow$excluded_invalid_ln, "\n")
cat("                       ▼\n")
cat("┌─────────────────────────────────────────────────┐\n")
cat("│  Valid LN records:    ", sprintf("%-20d", flow$step2_after_ln_valid), "│\n")
cat("└──────────────────────┬──────────────────────────┘\n")
cat("                       │\n")
cat("  Excluded: missing    │ n =", flow$excluded_missing_surv, "\n")
cat("  survival data        │\n")
cat("                       ▼\n")
cat("┌─────────────────────────────────────────────────┐\n")
cat("│  Complete survival:   ", sprintf("%-20d", flow$step3_after_surv), "│\n")
cat("└──────────────────────┬──────────────────────────┘\n")
cat("                       │\n")
cat("  Excluded: node-neg   │ n =", flow$excluded_node_negative, "\n")
cat("  (", NODE_POSITIVE_FILTER, ")             │\n", sep = "")
cat("                       ▼\n")
cat("┌─────────────────────────────────────────────────┐\n")
cat("│  Node-positive:      ", sprintf("%-20d", flow$step4_node_positive), "│\n")
cat("└──────────────────────┬──────────────────────────┘\n")
cat("                       │\n")
cat("  Excluded: TLE missing│ n =", flow$excluded_tle_missing, "\n")
cat("                       ▼\n")
cat("┌═════════════════════════════════════════════════┐\n")
cat("│  FINAL ANALYTIC COHORT: ", sprintf("%-18d", flow$step5_final_analytic), "│\n")
cat("│  OS events: ", sprintf("%-8d", n_os),
    "  DSS events: ", sprintf("%-8d", n_dss), "│\n")
cat("└═════════════════════════════════════════════════┘\n")

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 02 COMPLETE\n")
cat(strrep("=", 78), "\n")



  SCRIPT 02: DESCRIPTIVE STATISTICS (Phase 1)

Analytic cohort: N = 62 node-positive patients
Analytic tier: Inadequate (< 80 events) 


  TABLE 1: BASELINE CHARACTERISTICS

--- Table 1a: Stratified by LNR Tier ---
                              Stratified by LNR_tier
                               level            Overall                  
  n                                                  62                  
  AG (mean (SD))                                  60.26 (12.37)          
  Sex_f (%)                    Male                  19 (30.6)           
                               Female                43 (69.4)           
  T_stage_group (%)            T1/T2                  9 (14.5)           
                               T3                    42 (67.7)           
                               T4                    11 (17.7)           
  LN_total (median [IQR])                         15.00 [10.50, 21.75]   
  LN_positive (median [IQR])                       2.00 [1.00, 4.

Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_bar()`).”
Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_bar()`).”


Saved: Fig_LNR_histogram (PDF + PNG)

  OUTCOME SUMMARY

Overall Survival:
  Total patients: 62 
  Deaths (any cause): 34 ( 54.8 %)
  Censored: 28 ( 45.2 %)

Disease-Specific Survival:
  Cancer deaths: 24 ( 38.7 %)
  Non-cancer deaths: 10 

  CSS data completeness proxy: 70.6 %
  (Proportion of deceased patients with DSS event recorded)
  --> CSS analysis may be limited (< 85% threshold)

Median follow-up (reverse KM method): 2731 days ( 7.5 years)

  N-STAGE × LNR TIER CROSS-TABULATION

Counts:
    
     Low (<0.05) Intermediate (0.05-0.20) High (>0.20)
  N1           7                       27            7
  N2           0                        2           19

Row percentages (within each N-stage):
    
     Low (<0.05) Intermediate (0.05-0.20) High (>0.20)
  N1        17.1                     65.9         17.1
  N2         0.0                      9.5         90.5
Saved: Nstage_LNRtier_crosstab .csv

  CONSORT-STYLE PATIENT FLOW

┌─────────────────────────────────────────────────┐


## 3

In [ ]:
################################################################################
#  03_unadjusted_survival.R — Phase 2: Unadjusted Survival Analysis
#  Protocol Section 7.2 | SO1, SO2 foundational
#
#  Outputs: KM curves, log-rank tests, RMST, univariate Cox models,
#           LNR dose-response curve (martingale residual plot)
################################################################################

load(file.path(DIR_DATA, "analytic_dataset.RData"))

section_header("SCRIPT 03: UNADJUSTED SURVIVAL ANALYSIS (Phase 2)")

cat("Analytic cohort: N =", nrow(df), "| OS events:",
    sum(df$Outcome_OS == 1), "\n")
cat("Analytic tier:", ANALYTIC_TIER$label, "\n\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. KAPLAN-MEIER CURVES — OVERALL SURVIVAL BY N-STAGE
# ═══════════════════════════════════════════════════════════════════════════════

section_header("KM CURVES: OS BY N-STAGE")

# Fit KM
km_nstage <- survfit(Surv(Time_survival, Outcome_OS) ~ N_stage, data = df)
print(km_nstage)

# Log-rank test
lr_nstage <- survdiff(Surv(Time_survival, Outcome_OS) ~ N_stage, data = df)
cat("\nLog-rank test (N1 vs N2):\n")
print(lr_nstage)
lr_nstage_p <- 1 - pchisq(lr_nstage$chisq, df = length(lr_nstage$n) - 1)
cat("  p-value:", format_pval(lr_nstage_p), "\n")

# 1/3/5-year survival estimates
cat("\nSurvival estimates at landmark timepoints:\n")
surv_est_nstage <- summary(km_nstage, times = SURV_TIMEPOINTS_DAYS)
for (i in seq_along(SURV_TIMEPOINTS_YEARS)) {
  cat(sprintf("  %d-year survival:\n", SURV_TIMEPOINTS_YEARS[i]))
  idx <- which(surv_est_nstage$time == SURV_TIMEPOINTS_DAYS[i])
  if (length(idx) > 0) {
    for (j in idx) {
      cat(sprintf("    %s: %.1f%% (95%% CI: %.1f%% - %.1f%%)\n",
                  surv_est_nstage$strata[j],
                  surv_est_nstage$surv[j] * 100,
                  surv_est_nstage$lower[j] * 100,
                  surv_est_nstage$upper[j] * 100))
    }
  }
}

# KM plot
p_km_nstage <- ggsurvplot(
  km_nstage, data = df,
  palette = unname(COL_NSTAGE),
  risk.table = TRUE,
  risk.table.col = "strata",
  risk.table.height = 0.25,
  pval = TRUE,
  pval.method = TRUE,
  conf.int = TRUE,
  conf.int.alpha = 0.15,
  xlab = "Time (days)",
  ylab = "Overall Survival Probability",
  title = "Overall Survival by N-Stage",
  subtitle = "Node-positive CRC | Log-rank test",
  legend.title = "N-Stage",
  legend.labs = c("N1", "N2"),
  ggtheme = theme_publication(),
  break.time.by = 365.25,
  surv.median.line = "hv",
  xlim = c(0, max(df$Time_survival, na.rm = TRUE))
)
pdf(file.path(DIR_FIGURES, "Fig_KM_OS_by_Nstage.pdf"),
    width = FIG_WIDTH, height = FIG_HEIGHT + 1.5)
print(p_km_nstage)
dev.off()
png(file.path(DIR_FIGURES, "Fig_KM_OS_by_Nstage.png"),
    width = FIG_WIDTH, height = FIG_HEIGHT + 1.5, units = "in", res = FIG_DPI)
print(p_km_nstage)
dev.off()
cat("Saved: Fig_KM_OS_by_Nstage\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. KAPLAN-MEIER CURVES — OVERALL SURVIVAL BY LNR TIER
# ═══════════════════════════════════════════════════════════════════════════════

section_header("KM CURVES: OS BY LNR TIER")

km_lnr <- survfit(Surv(Time_survival, Outcome_OS) ~ LNR_tier, data = df)
print(km_lnr)

# Log-rank test
lr_lnr <- survdiff(Surv(Time_survival, Outcome_OS) ~ LNR_tier, data = df)
cat("\nLog-rank test (Low vs Intermediate vs High):\n")
print(lr_lnr)
lr_lnr_p <- 1 - pchisq(lr_lnr$chisq, df = length(lr_lnr$n) - 1)
cat("  p-value:", format_pval(lr_lnr_p), "\n")

# Pairwise log-rank with Bonferroni correction
cat("\nPairwise log-rank tests (Bonferroni corrected):\n")
if (nlevels(df$LNR_tier) > 1 && all(table(df$LNR_tier) > 0)) {
  pairwise_lr <- pairwise_survdiff(
    Surv(Time_survival, Outcome_OS) ~ LNR_tier,
    data = df, p.adjust.method = "bonferroni"
  )
  print(pairwise_lr)
}

# 1/3/5-year survival estimates
cat("\nSurvival estimates at landmark timepoints:\n")
surv_est_lnr <- summary(km_lnr, times = SURV_TIMEPOINTS_DAYS)
for (i in seq_along(SURV_TIMEPOINTS_YEARS)) {
  cat(sprintf("  %d-year survival:\n", SURV_TIMEPOINTS_YEARS[i]))
  idx <- which(surv_est_lnr$time == SURV_TIMEPOINTS_DAYS[i])
  if (length(idx) > 0) {
    for (j in idx) {
      cat(sprintf("    %s: %.1f%% (95%% CI: %.1f%% - %.1f%%)\n",
                  surv_est_lnr$strata[j],
                  surv_est_lnr$surv[j] * 100,
                  surv_est_lnr$lower[j] * 100,
                  surv_est_lnr$upper[j] * 100))
    }
  }
}

# KM plot
p_km_lnr <- ggsurvplot(
  km_lnr, data = df,
  palette = unname(COL_LNR_TIERS),
  risk.table = TRUE,
  risk.table.col = "strata",
  risk.table.height = 0.25,
  pval = TRUE,
  pval.method = TRUE,
  conf.int = TRUE,
  conf.int.alpha = 0.15,
  xlab = "Time (days)",
  ylab = "Overall Survival Probability",
  title = "Overall Survival by LNR Tier (Rosenberg)",
  subtitle = "Node-positive CRC | Thresholds: 0.05 / 0.20",
  legend.title = "LNR Tier",
  legend.labs = LNR_TIER_LABELS,
  ggtheme = theme_publication(),
  break.time.by = 365.25,
  surv.median.line = "hv",
  xlim = c(0, max(df$Time_survival, na.rm = TRUE))
)
pdf(file.path(DIR_FIGURES, "Fig_KM_OS_by_LNR_tier.pdf"),
    width = FIG_WIDTH, height = FIG_HEIGHT + 1.5)
print(p_km_lnr)
dev.off()
png(file.path(DIR_FIGURES, "Fig_KM_OS_by_LNR_tier.png"),
    width = FIG_WIDTH, height = FIG_HEIGHT + 1.5, units = "in", res = FIG_DPI)
print(p_km_lnr)
dev.off()
cat("Saved: Fig_KM_OS_by_LNR_tier\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. RESTRICTED MEAN SURVIVAL TIME (RMST) AT 5 YEARS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("RMST AT 5 YEARS")

# Determine truncation time — min of RMST_TAU_DAYS and max observed time
tau <- min(RMST_TAU_DAYS, max(df$Time_survival, na.rm = TRUE))
cat("RMST truncation time (tau):", round(tau, 0), "days (",
    round(tau / 365.25, 2), "years)\n\n")

# RMST by N-stage
cat("RMST by N-Stage:\n")
nstage_levels <- levels(df$N_stage)
for (lev in nstage_levels) {
  sub <- df[df$N_stage == lev, ]
  if (nrow(sub) > 0 && sum(sub$Outcome_OS == 1) > 0) {
    km_sub <- survfit(Surv(Time_survival, Outcome_OS) ~ 1, data = sub)
    # Compute RMST as area under KM curve up to tau
    rmst_val <- summary(km_sub, rmean = tau)$table["*rmean"]
    rmst_se <- summary(km_sub, rmean = tau)$table["*se(rmean)"]
    cat(sprintf("  %s: RMST = %.0f days (%.1f years) [SE: %.0f]\n",
                lev, rmst_val, rmst_val / 365.25, rmst_se))
  }
}

# RMST by LNR tier
cat("\nRMST by LNR Tier:\n")
lnr_levels <- levels(df$LNR_tier)
for (lev in lnr_levels) {
  sub <- df[df$LNR_tier == lev, ]
  if (nrow(sub) > 0 && sum(sub$Outcome_OS == 1) > 0) {
    km_sub <- survfit(Surv(Time_survival, Outcome_OS) ~ 1, data = sub)
    rmst_val <- summary(km_sub, rmean = tau)$table["*rmean"]
    rmst_se <- summary(km_sub, rmean = tau)$table["*se(rmean)"]
    cat(sprintf("  %s: RMST = %.0f days (%.1f years) [SE: %.0f]\n",
                lev, rmst_val, rmst_val / 365.25, rmst_se))
  } else {
    cat(sprintf("  %s: insufficient data (n = %d, events = %d)\n",
                lev, nrow(sub), sum(sub$Outcome_OS == 1)))
  }
}

# ═══════════════════════════════════════════════════════════════════════════════
# 4. CONTINUOUS LNR DOSE-RESPONSE (Martingale Residual Plot)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("LNR DOSE-RESPONSE ASSESSMENT")

# Martingale residuals from null Cox model
null_cox <- coxph(Surv(Time_survival, Outcome_OS) ~ 1, data = df)
df$mart_resid <- residuals(null_cox, type = "martingale")

p_dose_response <- ggplot(df, aes(x = LNR_prop, y = mart_resid)) +
  geom_point(alpha = 0.5, colour = "grey40", size = 2) +
  geom_smooth(method = "loess", se = TRUE, colour = "#B2182B",
              fill = "#B2182B", alpha = 0.15, linewidth = 1) +
  geom_hline(yintercept = 0, linetype = "dashed", colour = "grey60") +
  geom_vline(xintercept = c(LNR_CUT_LOW_INTER, LNR_CUT_INTER_HIGH),
             linetype = "dotted", colour = c("#4DAF4A", "#FF7F00"),
             linewidth = 0.6) +
  labs(title = "LNR Dose-Response: Martingale Residuals from Null Cox Model",
       subtitle = "LOESS smoothed | Rosenberg thresholds indicated",
       x = "Lymph Node Ratio (proportion)",
       y = "Martingale Residual") +
  theme_publication()
save_figure(p_dose_response, "Fig_LNR_dose_response_martingale")

cat("Visual assessment: Check whether the LOESS curve shows a\n")
cat("monotonically increasing relationship (linear dose-response)\n")
cat("or a non-linear (threshold/plateau) pattern.\n")
cat("This determines whether RCS parametrisation is justified.\n\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. UNIVARIATE COX MODELS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("UNIVARIATE COX MODELS")

# --- 5.1 N-Stage (layer 1, Model A) ---
cat("--- Model A: N-Stage (univariate) ---\n")
cox_nstage_uni <- coxph(Surv(Time_survival, Outcome_OS) ~ N_stage, data = df)
print(summary(cox_nstage_uni))

# --- 5.2 LNR continuous (linear) ---
cat("\n--- LNR continuous (linear, univariate) ---\n")
cox_lnr_linear <- coxph(Surv(Time_survival, Outcome_OS) ~ LNR_prop, data = df)
print(summary(cox_lnr_linear))

# --- 5.3 LNR continuous with RCS (Layer 1, Model B) ---
# Only if tier permits RCS
if (ANALYTIC_TIER$permit_rcs || ANALYTIC_TIER$tier_code >= 2) {
  cat("\n--- Model B: LNR continuous (RCS, 3 knots, univariate) ---\n")
  # Set data distribution for rms
  dd <- datadist(df)
  options(datadist = "dd")

  cox_lnr_rcs <- cph(Surv(Time_survival, Outcome_OS) ~ rcs(LNR_prop, 3),
                      data = df, x = TRUE, y = TRUE, surv = TRUE)
  print(cox_lnr_rcs)
  cat("\nAnova (non-linearity test):\n")
  print(anova(cox_lnr_rcs))

  # RCS association plot (log-HR vs LNR)
  p_rcs <- ggplot(Predict(cox_lnr_rcs, LNR_prop, fun = exp),
                  aes(x = LNR_prop, y = yhat)) +
    geom_line(colour = "#B2182B", linewidth = 1) +
    geom_ribbon(aes(ymin = lower, ymax = upper), alpha = 0.15, fill = "#B2182B") +
    geom_hline(yintercept = 1, linetype = "dashed", colour = "grey50") +
    labs(title = "LNR–Mortality Association (RCS, 3 knots)",
         subtitle = "Hazard ratio relative to reference LNR",
         x = "Lymph Node Ratio (proportion)",
         y = "Hazard Ratio") +
    theme_publication()
  save_figure(p_rcs, "Fig_LNR_RCS_association_plot")
} else {
  cat("\n  RCS not permitted at this analytic tier. Using linear LNR.\n")
  # For consistency, define Model B as linear LNR
  dd <- datadist(df)
  options(datadist = "dd")
  cox_lnr_rcs <- cph(Surv(Time_survival, Outcome_OS) ~ LNR_prop,
                      data = df, x = TRUE, y = TRUE, surv = TRUE)
  print(cox_lnr_rcs)
}

# --- 5.4 LNR categorical tiers ---
cat("\n--- LNR categorical tiers (Rosenberg, univariate) ---\n")
cox_lnr_cat <- coxph(Surv(Time_survival, Outcome_OS) ~ LNR_tier, data = df)
print(summary(cox_lnr_cat))

# --- 5.5 All other covariates (univariate) ---
cat("\n--- Other covariates (univariate) ---\n")
uni_vars <- c("AG", "Sex_f", "T_stage_group", "Grade_f", "LVI_f",
              "LN_total", "TLE_adequate")

uni_results <- data.frame(
  Variable = character(),
  HR = numeric(),
  HR_lower = numeric(),
  HR_upper = numeric(),
  p_value = numeric(),
  stringsAsFactors = FALSE
)

for (v in uni_vars) {
  formula_v <- as.formula(paste("Surv(Time_survival, Outcome_OS) ~", v))
  fit_v <- tryCatch(
    coxph(formula_v, data = df),
    error = function(e) NULL
  )
  if (!is.null(fit_v)) {
    s <- summary(fit_v)
    for (r in seq_len(nrow(s$coefficients))) {
      uni_results <- rbind(uni_results, data.frame(
        Variable = rownames(s$coefficients)[r],
        HR = round(s$conf.int[r, "exp(coef)"], 3),
        HR_lower = round(s$conf.int[r, "lower .95"], 3),
        HR_upper = round(s$conf.int[r, "upper .95"], 3),
        p_value = round(s$coefficients[r, "Pr(>|z|)"], 4),
        stringsAsFactors = FALSE
      ))
    }
  }
}

# Add LNR and N-stage
s_ns <- summary(cox_nstage_uni)
for (r in seq_len(nrow(s_ns$coefficients))) {
  uni_results <- rbind(uni_results, data.frame(
    Variable = rownames(s_ns$coefficients)[r],
    HR = round(s_ns$conf.int[r, "exp(coef)"], 3),
    HR_lower = round(s_ns$conf.int[r, "lower .95"], 3),
    HR_upper = round(s_ns$conf.int[r, "upper .95"], 3),
    p_value = round(s_ns$coefficients[r, "Pr(>|z|)"], 4),
    stringsAsFactors = FALSE
  ))
}

s_lnr <- summary(cox_lnr_linear)
uni_results <- rbind(uni_results, data.frame(
  Variable = "LNR_prop (continuous)",
  HR = round(s_lnr$conf.int[1, "exp(coef)"], 3),
  HR_lower = round(s_lnr$conf.int[1, "lower .95"], 3),
  HR_upper = round(s_lnr$conf.int[1, "upper .95"], 3),
  p_value = round(s_lnr$coefficients[1, "Pr(>|z|)"], 4),
  stringsAsFactors = FALSE
))

cat("\nUnivariate Cox regression — all covariates:\n")
print(uni_results)
save_table(uni_results, "univariate_cox_results")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. DISEASE-SPECIFIC SURVIVAL — KM (if applicable)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("KM CURVES: DSS (DISEASE-SPECIFIC SURVIVAL)")

n_dss_events <- sum(df$Outcome_DSS == 1, na.rm = TRUE)
cat("DSS events:", n_dss_events, "\n")

if (n_dss_events >= 5) {
  # KM by N-stage
  km_dss_nstage <- survfit(Surv(Time_survival, Outcome_DSS) ~ N_stage, data = df)
  p_km_dss_ns <- ggsurvplot(
    km_dss_nstage, data = df, palette = unname(COL_NSTAGE),
    risk.table = TRUE, risk.table.height = 0.25,
    pval = TRUE, pval.method = TRUE, conf.int = TRUE, conf.int.alpha = 0.15,
    xlab = "Time (days)", ylab = "Disease-Specific Survival Probability",
    title = "Disease-Specific Survival by N-Stage",
    legend.title = "N-Stage", legend.labs = c("N1", "N2"),
    ggtheme = theme_publication(), break.time.by = 365.25
  )
  pdf(file.path(DIR_FIGURES, "Fig_KM_DSS_by_Nstage.pdf"),
      width = FIG_WIDTH, height = FIG_HEIGHT + 1.5)
  print(p_km_dss_ns)
  dev.off()
  png(file.path(DIR_FIGURES, "Fig_KM_DSS_by_Nstage.png"),
      width = FIG_WIDTH, height = FIG_HEIGHT + 1.5, units = "in", res = FIG_DPI)
  print(p_km_dss_ns)
  dev.off()

  # KM by LNR tier
  km_dss_lnr <- survfit(Surv(Time_survival, Outcome_DSS) ~ LNR_tier, data = df)
  p_km_dss_lnr <- ggsurvplot(
    km_dss_lnr, data = df, palette = unname(COL_LNR_TIERS),
    risk.table = TRUE, risk.table.height = 0.25,
    pval = TRUE, pval.method = TRUE, conf.int = TRUE, conf.int.alpha = 0.15,
    xlab = "Time (days)", ylab = "Disease-Specific Survival Probability",
    title = "Disease-Specific Survival by LNR Tier",
    legend.title = "LNR Tier", legend.labs = LNR_TIER_LABELS,
    ggtheme = theme_publication(), break.time.by = 365.25
  )
  pdf(file.path(DIR_FIGURES, "Fig_KM_DSS_by_LNR_tier.pdf"),
      width = FIG_WIDTH, height = FIG_HEIGHT + 1.5)
  print(p_km_dss_lnr)
  dev.off()
  png(file.path(DIR_FIGURES, "Fig_KM_DSS_by_LNR_tier.png"),
      width = FIG_WIDTH, height = FIG_HEIGHT + 1.5, units = "in", res = FIG_DPI)
  print(p_km_dss_lnr)
  dev.off()

  cat("Saved: Fig_KM_DSS_by_Nstage, Fig_KM_DSS_by_LNR_tier\n")
} else {
  cat("  Insufficient DSS events for KM analysis.\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 7. SAVE SESSION OBJECTS
# ═══════════════════════════════════════════════════════════════════════════════

save(km_nstage, km_lnr, cox_nstage_uni, cox_lnr_linear, cox_lnr_rcs,
     cox_lnr_cat, uni_results, lr_nstage_p, lr_lnr_p,
     file = file.path(DIR_MODELS, "phase2_unadjusted_results.RData"))

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 03 COMPLETE\n")
cat(strrep("=", 78), "\n")



  SCRIPT 03: UNADJUSTED SURVIVAL ANALYSIS (Phase 2)

Analytic cohort: N = 62 | OS events: 34 
Analytic tier: Inadequate (< 80 events) 


  KM CURVES: OS BY N-STAGE

Call: survfit(formula = Surv(Time_survival, Outcome_OS) ~ N_stage, 
    data = df)

            n events median 0.95LCL 0.95UCL
N_stage=N1 41     19   3067     924      NA
N_stage=N2 21     15    705     298      NA

Log-rank test (N1 vs N2):
Call:
survdiff(formula = Surv(Time_survival, Outcome_OS) ~ N_stage, 
    data = df)

            N Observed Expected (O-E)^2/E (O-E)^2/V
N_stage=N1 41       19    24.25      1.14      3.99
N_stage=N2 21       15     9.75      2.83      3.99

 Chisq= 4  on 1 degrees of freedom, p= 0.05 
  p-value: 0.046 

Survival estimates at landmark timepoints:
  1-year survival:
    N_stage=N1: 84.7% (95% CI: 74.2% - 96.8%)
    N_stage=N2: 61.2% (95% CI: 43.4% - 86.4%)
  3-year survival:
    N_stage=N1: 56.5% (95% CI: 42.9% - 74.4%)
    N_stage=N2: 40.8% (95% CI: 24.1% - 69.2%)
  5-year survival:
 

agg_record_9db72ce4091f 
                      2

agg_record_9db72ce4091f 
                      2

Saved: Fig_KM_OS_by_Nstage

  KM CURVES: OS BY LNR TIER

Call: survfit(formula = Surv(Time_survival, Outcome_OS) ~ LNR_tier, 
    data = df)

                                   n events median 0.95LCL 0.95UCL
LNR_tier=Low (<0.05)               7      4   1131     809      NA
LNR_tier=Intermediate (0.05-0.20) 29     15   1380     618      NA
LNR_tier=High (>0.20)             26     15   1060     526      NA

Log-rank test (Low vs Intermediate vs High):
Call:
survdiff(formula = Surv(Time_survival, Outcome_OS) ~ LNR_tier, 
    data = df)

                                   N Observed Expected (O-E)^2/E (O-E)^2/V
LNR_tier=Low (<0.05)               7        4     4.26    0.0164    0.0189
LNR_tier=Intermediate (0.05-0.20) 29       15    15.92    0.0531    0.1004
LNR_tier=High (>0.20)             26       15    13.82    0.1014    0.1711

 Chisq= 0.2  on 2 degrees of freedom, p= 0.9 
  p-value: 0.918 

Pairwise log-rank tests (Bonferroni corrected):

	Pairwise comparisons using Log-Rank test 


agg_record_9db72ce4091f 
                      2

...

Saved: Fig_KM_OS_by_LNR_tier

  RMST AT 5 YEARS

RMST truncation time (tau): 1826 days ( 5 years)

RMST by N-Stage:
  N1: RMST = NA days (NA years) [SE: NA]
  N2: RMST = NA days (NA years) [SE: NA]

RMST by LNR Tier:
  Low (<0.05): RMST = NA days (NA years) [SE: NA]
  Intermediate (0.05-0.20): RMST = NA days (NA years) [SE: NA]
  High (>0.20): RMST = NA days (NA years) [SE: NA]

  LNR DOSE-RESPONSE ASSESSMENT



`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'


Saved: Fig_LNR_dose_response_martingale (PDF + PNG)
Visual assessment: Check whether the LOESS curve shows a
monotonically increasing relationship (linear dose-response)
or a non-linear (threshold/plateau) pattern.
This determines whether RCS parametrisation is justified.


  UNIVARIATE COX MODELS

--- Model A: N-Stage (univariate) ---
Call:
coxph(formula = Surv(Time_survival, Outcome_OS) ~ N_stage, data = df)

  n= 62, number of events= 34 

            coef exp(coef) se(coef)     z Pr(>|z|)  
N_stageN2 0.6797    1.9732   0.3461 1.964   0.0496 *
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

          exp(coef) exp(-coef) lower .95 upper .95
N_stageN2     1.973     0.5068     1.001     3.889

Concordance= 0.586  (se = 0.042 )
Likelihood ratio test= 3.69  on 1 df,   p=0.05
Wald test            = 3.86  on 1 df,   p=0.05
Score (logrank) test = 4  on 1 df,   p=0.05


--- LNR continuous (linear, univariate) ---
Call:
coxph(formula = Surv(Time_survival, Outcome_OS) ~ LN

Ignoring unknown labels:
• colour : "N-Stage"
Ignoring unknown labels:
• colour : "N-Stage"
Ignoring unknown labels:
• colour : "LNR Tier"
Ignoring unknown labels:
• colour : "LNR Tier"


Saved: Fig_KM_DSS_by_Nstage, Fig_KM_DSS_by_LNR_tier

  SCRIPT 03 COMPLETE


## 4

In [ ]:
################################################################################
#  04_cox_models.R — Phase 3: Three-Layer Cox Model Comparison
#  Protocol Section 7.3 | SO2, SO3
#
#  Implements the six-model symmetric comparison framework:
#    Layer 1 (Unadjusted): Model A (N-stage) vs Model B (LNR RCS)
#    Layer 2 (TLE-adjusted): Model C2 vs Model D2
#    Layer 3 (Full MVA):     Model C vs Model D  (PRIMARY)
#
#  Model complexity adapts automatically to the event-count gating tier.
################################################################################

load(file.path(DIR_DATA, "analytic_dataset.RData"))

section_header("SCRIPT 04: THREE-LAYER COX MODEL COMPARISON (Phase 3)")

cat("Analytic cohort: N =", nrow(df), "| OS events:",
    sum(df$Outcome_OS == 1), "\n")
cat("Analytic tier:", ANALYTIC_TIER$label, "\n")
cat("Permit MVA:", ANALYTIC_TIER$permit_mva,
    "| Permit RCS:", ANALYTIC_TIER$permit_rcs, "\n\n")

# Set up rms data distribution
dd <- datadist(df)
options(datadist = "dd")

# Store all models in a named list
models <- list()
model_summary <- data.frame(
  Model = character(),
  Layer = character(),
  Description = character(),
  df_used = integer(),
  AIC = numeric(),
  BIC = numeric(),
  LogLik = numeric(),
  stringsAsFactors = FALSE
)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LAYER 1 — UNADJUSTED MODELS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("LAYER 1: UNADJUSTED MODELS")

# --- Model A: N-stage only ---
cat("━━━ Model A: N-stage only ━━━\n")
models$A <- cph(Surv(Time_survival, Outcome_OS) ~ N_stage,
                data = df, x = TRUE, y = TRUE, surv = TRUE)
print(models$A)
cat("\n")

model_summary <- rbind(model_summary, data.frame(
  Model = "A", Layer = "1 (Unadjusted)",
  Description = "N-stage only",
  df_used = models$A$stats["d.f."],
  AIC = AIC(models$A), BIC = BIC(models$A),
  LogLik = models$A$loglik[2]
))

# --- Model B: LNR (continuous) ---
cat("━━━ Model B: LNR continuous ━━━\n")
if (ANALYTIC_TIER$permit_rcs) {
  models$B <- cph(Surv(Time_survival, Outcome_OS) ~ rcs(LNR_prop, 3),
                  data = df, x = TRUE, y = TRUE, surv = TRUE)
  cat("  (RCS with 3 knots)\n")
} else {
  models$B <- cph(Surv(Time_survival, Outcome_OS) ~ LNR_prop,
                  data = df, x = TRUE, y = TRUE, surv = TRUE)
  cat("  (Linear — RCS not permitted at this tier)\n")
}
print(models$B)
cat("\nAnova:\n")
print(anova(models$B))

model_summary <- rbind(model_summary, data.frame(
  Model = "B", Layer = "1 (Unadjusted)",
  Description = ifelse(ANALYTIC_TIER$permit_rcs, "LNR RCS(3)", "LNR linear"),
  df_used = models$B$stats["d.f."],
  AIC = AIC(models$B), BIC = BIC(models$B),
  LogLik = models$B$loglik[2]
))

# ═══════════════════════════════════════════════════════════════════════════════
# 2. LAYER 2 — TLE-ADJUSTED MODELS (if MVA permitted)
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_mva) {
  section_header("LAYER 2: TLE-ADJUSTED MODELS")

  # --- Model C2: N-stage + TLE ---
  cat("━━━ Model C2: N-stage + TLE ━━━\n")
  if (ANALYTIC_TIER$permit_rcs) {
    models$C2 <- cph(Surv(Time_survival, Outcome_OS) ~ N_stage + rcs(LN_total, 3),
                     data = df, x = TRUE, y = TRUE, surv = TRUE)
    cat("  (TLE modelled with RCS, 3 knots)\n")
  } else {
    models$C2 <- cph(Surv(Time_survival, Outcome_OS) ~ N_stage + LN_total,
                     data = df, x = TRUE, y = TRUE, surv = TRUE)
    cat("  (TLE modelled linearly)\n")
  }
  print(models$C2)
  cat("\nAnova:\n")
  print(anova(models$C2))

  model_summary <- rbind(model_summary, data.frame(
    Model = "C2", Layer = "2 (TLE-adjusted)",
    Description = ifelse(ANALYTIC_TIER$permit_rcs,
                         "N-stage + TLE RCS(3)", "N-stage + TLE linear"),
    df_used = models$C2$stats["d.f."],
    AIC = AIC(models$C2), BIC = BIC(models$C2),
    LogLik = models$C2$loglik[2]
  ))

  # --- Model D2: LNR + TLE ---
  cat("\n━━━ Model D2: LNR + TLE ━━━\n")
  if (ANALYTIC_TIER$permit_rcs) {
    models$D2 <- cph(Surv(Time_survival, Outcome_OS) ~
                       rcs(LNR_prop, 3) + rcs(LN_total, 3),
                     data = df, x = TRUE, y = TRUE, surv = TRUE)
    cat("  (Both LNR and TLE modelled with RCS, 3 knots each)\n")
  } else {
    models$D2 <- cph(Surv(Time_survival, Outcome_OS) ~ LNR_prop + LN_total,
                     data = df, x = TRUE, y = TRUE, surv = TRUE)
    cat("  (Both LNR and TLE modelled linearly)\n")
  }
  print(models$D2)
  cat("\nAnova:\n")
  print(anova(models$D2))

  model_summary <- rbind(model_summary, data.frame(
    Model = "D2", Layer = "2 (TLE-adjusted)",
    Description = ifelse(ANALYTIC_TIER$permit_rcs,
                         "LNR RCS(3) + TLE RCS(3)", "LNR linear + TLE linear"),
    df_used = models$D2$stats["d.f."],
    AIC = AIC(models$D2), BIC = BIC(models$D2),
    LogLik = models$D2$loglik[2]
  ))

} else {
  cat("\n  Layer 2 SKIPPED — MVA not permitted at Tier",
      ANALYTIC_TIER$tier_code, "\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 3. LAYER 3 — FULL MVA MODELS (PRIMARY COMPARISON)
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_mva) {
  section_header("LAYER 3: FULL MVA MODELS (PRIMARY)")

  # Define covariate block — forced entry, no stepwise selection (per protocol)
  # Available covariates: AG, Sex_f, T_stage_group, Grade_f, LVI_f, LN_total
  # Missing from protocol: PNI, resection margin, tumour site, calendar period

  # --- Model C: N-stage + TLE + all covariates (Primary N-stage model) ---
  cat("━━━ Model C: N-stage Full MVA (PRIMARY) ━━━\n")
  cat("  Covariates: N-stage + TLE + Age + Sex + T-stage + Grade + LVI\n")
  cat("  (PNI, margin, site, calendar period NOT available in dataset)\n\n")

  if (ANALYTIC_TIER$permit_rcs) {
    models$C <- cph(Surv(Time_survival, Outcome_OS) ~
                      N_stage + rcs(LN_total, 3) + AG + Sex_f +
                      T_stage_group + Grade_f + LVI_f,
                    data = df, x = TRUE, y = TRUE, surv = TRUE)
  } else {
    models$C <- cph(Surv(Time_survival, Outcome_OS) ~
                      N_stage + LN_total + AG + Sex_f +
                      T_stage_group + Grade_f + LVI_f,
                    data = df, x = TRUE, y = TRUE, surv = TRUE)
  }
  print(models$C)
  cat("\nAnova:\n")
  print(anova(models$C))

  model_summary <- rbind(model_summary, data.frame(
    Model = "C", Layer = "3 (Full MVA)",
    Description = "N-stage + all covariates (PRIMARY)",
    df_used = models$C$stats["d.f."],
    AIC = AIC(models$C), BIC = BIC(models$C),
    LogLik = models$C$loglik[2]
  ))

  # --- Model D: LNR + TLE + all covariates (Primary LNR model) ---
  cat("\n━━━ Model D: LNR Full MVA (PRIMARY) ━━━\n")
  cat("  Covariates: LNR + TLE + Age + Sex + T-stage + Grade + LVI\n\n")

  if (ANALYTIC_TIER$permit_rcs) {
    models$D <- cph(Surv(Time_survival, Outcome_OS) ~
                      rcs(LNR_prop, 3) + rcs(LN_total, 3) + AG + Sex_f +
                      T_stage_group + Grade_f + LVI_f,
                    data = df, x = TRUE, y = TRUE, surv = TRUE)
  } else {
    models$D <- cph(Surv(Time_survival, Outcome_OS) ~
                      LNR_prop + LN_total + AG + Sex_f +
                      T_stage_group + Grade_f + LVI_f,
                    data = df, x = TRUE, y = TRUE, surv = TRUE)
  }
  print(models$D)
  cat("\nAnova:\n")
  print(anova(models$D))

  model_summary <- rbind(model_summary, data.frame(
    Model = "D", Layer = "3 (Full MVA)",
    Description = "LNR + all covariates (PRIMARY)",
    df_used = models$D$stats["d.f."],
    AIC = AIC(models$D), BIC = BIC(models$D),
    LogLik = models$D$loglik[2]
  ))

} else {
  cat("\n  Layer 3 SKIPPED — MVA not permitted at Tier",
      ANALYTIC_TIER$tier_code, "\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 4. INTERACTION MODEL (Only if Tier >= 4, i.e. >= 200 events)
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_interaction) {
  section_header("INTERACTION MODEL: LNR × TLE ADEQUACY (H4)")

  if (ANALYTIC_TIER$permit_rcs) {
    models$D_interaction <- cph(
      Surv(Time_survival, Outcome_OS) ~
        rcs(LNR_prop, 3) * TLE_adequate_binary + rcs(LN_total, 3) +
        AG + Sex_f + T_stage_group + Grade_f + LVI_f,
      data = df, x = TRUE, y = TRUE, surv = TRUE
    )
  } else {
    models$D_interaction <- cph(
      Surv(Time_survival, Outcome_OS) ~
        LNR_prop * TLE_adequate_binary + LN_total +
        AG + Sex_f + T_stage_group + Grade_f + LVI_f,
      data = df, x = TRUE, y = TRUE, surv = TRUE
    )
  }
  print(models$D_interaction)
  cat("\nAnova (interaction test):\n")
  print(anova(models$D_interaction))

} else {
  cat("\n  Interaction model NOT permitted at Tier",
      ANALYTIC_TIER$tier_code, "(requires >= 200 events)\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 5. PROPORTIONAL HAZARDS ASSUMPTION TESTING
# ═══════════════════════════════════════════════════════════════════════════════

section_header("PROPORTIONAL HAZARDS ASSUMPTION TESTING")

# Test PH for all built models using survival::cox.zph on coxph equivalent
# cph objects from rms can be tested similarly
for (mname in names(models)) {
  cat(sprintf("\n━━━ PH test for Model %s ━━━\n", mname))

  # Convert to coxph if needed for cox.zph
  tryCatch({
    ph_test <- cox.zph(models[[mname]], transform = "log")
    print(ph_test)

    # Check for violations
    violations <- ph_test$table[, "p"] < PH_VIOLATION_THRESHOLD
    if (any(violations[-length(violations)])) {  # Exclude GLOBAL
      violated_vars <- rownames(ph_test$table)[which(violations[-length(violations)])]
      cat("  *** PH VIOLATION detected for:",
          paste(violated_vars, collapse = ", "), "\n")
      cat("  --> Per protocol: consider stratification or time-interaction term.\n")
    } else {
      cat("  No PH violations detected (all p >", PH_VIOLATION_THRESHOLD, ").\n")
    }

    # Save Schoenfeld residual plots for primary models (C and D)
    if (mname %in% c("C", "D")) {
      pdf(file.path(DIR_FIGURES, paste0("Fig_PH_test_Model_", mname, ".pdf")),
          width = FIG_WIDTH, height = FIG_HEIGHT)
      plot(ph_test, main = paste("Schoenfeld Residuals — Model", mname))
      dev.off()
      cat("  Saved: Fig_PH_test_Model_", mname, ".pdf\n")
    }
  }, error = function(e) {
    cat("  PH test error:", e$message, "\n")
  })
}

# ═══════════════════════════════════════════════════════════════════════════════
# 6. MODEL COMPARISON SUMMARY TABLE
# ═══════════════════════════════════════════════════════════════════════════════

section_header("MODEL COMPARISON SUMMARY")

cat("Model summary across all layers:\n")
print(model_summary)
save_table(model_summary, "model_comparison_summary")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. ASSOCIATION PLOT — LOG-HR VS LNR (FROM RCS MODEL)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("ASSOCIATION PLOTS")

# From the primary Model D (if available)
if (!is.null(models$D)) {
  pred_lnr <- Predict(models$D, LNR_prop, fun = exp)

  p_assoc <- ggplot(pred_lnr, aes(x = LNR_prop, y = yhat)) +
    geom_line(colour = "#B2182B", linewidth = 1.2) +
    geom_ribbon(aes(ymin = lower, ymax = upper),
                fill = "#B2182B", alpha = 0.12) +
    geom_hline(yintercept = 1, linetype = "dashed", colour = "grey50") +
    geom_vline(xintercept = c(LNR_CUT_LOW_INTER, LNR_CUT_INTER_HIGH),
               linetype = "dotted", colour = c("#4DAF4A", "#FF7F00"),
               linewidth = 0.6) +
    labs(title = "LNR–Mortality Association (Full MVA, Model D)",
         subtitle = "Hazard ratio relative to reference | Adjusted for all covariates",
         x = "Lymph Node Ratio (proportion)",
         y = "Hazard Ratio (HR)") +
    theme_publication()
  save_figure(p_assoc, "Fig_LNR_HR_association_fullMVA")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 8. FOREST PLOT — FULL MVA HAZARD RATIOS
# ═══════════════════════════════════════════════════════════════════════════════

if (!is.null(models$D)) {
  section_header("FOREST PLOT: MODEL D (LNR FULL MVA)")

  # Extract coefficients from coxph-equivalent for forest plot
  cox_D <- coxph(Surv(Time_survival, Outcome_OS) ~
                   LNR_prop + LN_total + AG + Sex_f +
                   T_stage_group + Grade_f + LVI_f,
                 data = df)

  forest_data <- broom::tidy(cox_D, exponentiate = TRUE, conf.int = TRUE)
  forest_data$term <- gsub("Sex_f", "Sex: ", forest_data$term)
  forest_data$term <- gsub("T_stage_group", "T-stage: ", forest_data$term)
  forest_data$term <- gsub("Grade_f", "Grade: ", forest_data$term)
  forest_data$term <- gsub("LVI_f", "LVI: ", forest_data$term)

  p_forest <- ggplot(forest_data, aes(x = estimate, y = reorder(term, estimate))) +
    geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
    geom_point(size = 3, colour = "#B2182B") +
    geom_errorbarh(aes(xmin = conf.low, xmax = conf.high),
                   height = 0.2, colour = "#B2182B") +
    labs(title = "Forest Plot: Model D (LNR Full MVA)",
         subtitle = "Hazard ratios with 95% CI for Overall Survival",
         x = "Hazard Ratio (log scale)",
         y = NULL) +
    scale_x_log10() +
    theme_publication() +
    theme(axis.text.y = element_text(size = 10))
  save_figure(p_forest, "Fig_forest_plot_ModelD", height = 5)
}

# ═══════════════════════════════════════════════════════════════════════════════
# 9. SAVE ALL MODEL OBJECTS
# ═══════════════════════════════════════════════════════════════════════════════

save(models, model_summary, dd,
     file = file.path(DIR_MODELS, "phase3_cox_models.RData"))

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 04 COMPLETE\n")
cat("  Models built:", paste(names(models), collapse = ", "), "\n")
cat(strrep("=", 78), "\n")



  SCRIPT 04: THREE-LAYER COX MODEL COMPARISON (Phase 3)

Analytic cohort: N = 62 | OS events: 34 
Analytic tier: Inadequate (< 80 events) 
Permit MVA: FALSE | Permit RCS: FALSE 


  LAYER 1: UNADJUSTED MODELS

━━━ Model A: N-stage only ━━━
Cox Proportional Hazards Model

cph(formula = Surv(Time_survival, Outcome_OS) ~ N_stage, data = df, 
    x = TRUE, y = TRUE, surv = TRUE)

                       Model Tests    Discrimination    
                                             Indexes    
Obs        62    LR chi2      3.69    R2       0.059    
Events     34    d.f.            1    R2(1,62) 0.042    
Center 0.2302    Pr(> chi2) 0.0547    R2(1,34) 0.076    
                 Score chi2   4.00    Dxy      0.172    
                 Pr(> chi2) 0.0454                      

           Coef   S.E.   Wald Z Pr(>|Z|)
N_stage=N2 0.6797 0.3461 1.96   0.0496  


━━━ Model B: LNR continuous ━━━
  (Linear — RCS not permitted at this tier)
Cox Proportional Hazards Model

cph(formula = Surv(Time_surv

## 5

In [ ]:
################################################################################
#  05_performance_metrics.R — Phase 3: Performance Metrics
#  Protocol Section 7.4 | SO2
#
#  Primary co-endpoints: Uno C-statistic + Decision-Curve Analysis
#  Primary secondary: Calibration + AIC/BIC
#  Exploratory: NRI, IDI (appendix only)
#  Internal validation: Bootstrap optimism-corrected C-statistic
################################################################################

load(file.path(DIR_DATA, "analytic_dataset.RData"))
load(file.path(DIR_MODELS, "phase3_cox_models.RData"))

section_header("SCRIPT 05: PERFORMANCE METRICS (Phase 3)")

cat("Analytic tier:", ANALYTIC_TIER$label, "\n\n")

# Re-establish rms datadist
dd <- datadist(df)
options(datadist = "dd")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. UNO'S C-STATISTIC FOR ALL MODELS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("UNO'S C-STATISTIC")

# Truncation time for Uno's C
tau_uno <- quantile(df$Time_survival[df$Outcome_OS == 1],
                    UNO_TRUNCATION_QUANTILE, na.rm = TRUE)
cat("Uno C-stat truncation (75th percentile of event times):",
    round(tau_uno, 0), "days\n\n")

# Function to compute Uno C with bootstrap CI
compute_uno_c <- function(model_name, model_obj, data) {
  # Get linear predictor
  lp <- tryCatch(predict(model_obj, newdata = data, type = "lp"),
                 error = function(e) predict(model_obj, type = "lp"))

  # Uno's concordance via survival::concordance
  conc <- concordance(Surv(Time_survival, Outcome_OS) ~ lp,
                      data = data, timewt = "n/G2")

  c_stat <- conc$concordance
  c_se <- sqrt(conc$var)
  c_lower <- c_stat - 1.96 * c_se
  c_upper <- c_stat + 1.96 * c_se

  data.frame(
    Model = model_name,
    C_statistic = round(c_stat, 4),
    SE = round(c_se, 4),
    CI_lower = round(c_lower, 4),
    CI_upper = round(c_upper, 4),
    stringsAsFactors = FALSE
  )
}

# Compute C for all models
c_results <- data.frame()
for (mname in names(models)) {
  tryCatch({
    c_row <- compute_uno_c(mname, models[[mname]], df)
    c_results <- rbind(c_results, c_row)
    cat(sprintf("  Model %s: C = %.4f (95%% CI: %.4f - %.4f)\n",
                c_row$Model, c_row$C_statistic, c_row$CI_lower, c_row$CI_upper))
  }, error = function(e) {
    cat(sprintf("  Model %s: C-stat computation failed: %s\n", mname, e$message))
  })
}

cat("\nC-statistic summary table:\n")
print(c_results)
save_table(c_results, "C_statistic_all_models")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. PRIMARY COMPARISON: ΔC BETWEEN MODEL D AND MODEL C
# ═══════════════════════════════════════════════════════════════════════════════

section_header("PRIMARY ΔC COMPARISON (MODEL D vs MODEL C)")

if (!is.null(models$C) && !is.null(models$D)) {

  c_model_C <- c_results$C_statistic[c_results$Model == "C"]
  c_model_D <- c_results$C_statistic[c_results$Model == "D"]
  delta_c <- c_model_D - c_model_C

  cat("Model C (N-stage Full MVA) C-statistic:", c_model_C, "\n")
  cat("Model D (LNR Full MVA) C-statistic:", c_model_D, "\n")
  cat("ΔC (Model D - Model C):", round(delta_c, 4), "\n\n")

  # Bootstrap ΔC with BCa CI
  cat("Computing bootstrap 95% CI for ΔC (", N_BOOTSTRAP, " resamples)...\n")

  boot_delta_c <- function(data, indices) {
    boot_data <- data[indices, ]

    # Refit both models on bootstrap sample
    tryCatch({
      if (ANALYTIC_TIER$permit_rcs) {
        dd_boot <- datadist(boot_data)
        options(datadist = "dd_boot")

        fit_C <- cph(Surv(Time_survival, Outcome_OS) ~
                       N_stage + rcs(LN_total, 3) + AG + Sex_f +
                       T_stage_group + Grade_f + LVI_f,
                     data = boot_data, x = TRUE, y = TRUE)
        fit_D <- cph(Surv(Time_survival, Outcome_OS) ~
                       rcs(LNR_prop, 3) + rcs(LN_total, 3) + AG + Sex_f +
                       T_stage_group + Grade_f + LVI_f,
                     data = boot_data, x = TRUE, y = TRUE)
      } else {
        fit_C <- cph(Surv(Time_survival, Outcome_OS) ~
                       N_stage + LN_total + AG + Sex_f +
                       T_stage_group + Grade_f + LVI_f,
                     data = boot_data, x = TRUE, y = TRUE)
        fit_D <- cph(Surv(Time_survival, Outcome_OS) ~
                       LNR_prop + LN_total + AG + Sex_f +
                       T_stage_group + Grade_f + LVI_f,
                     data = boot_data, x = TRUE, y = TRUE)
      }

      lp_C <- predict(fit_C, type = "lp")
      lp_D <- predict(fit_D, type = "lp")

      conc_C <- concordance(Surv(Time_survival, Outcome_OS) ~ lp_C,
                            data = boot_data, timewt = "n/G2")
      conc_D <- concordance(Surv(Time_survival, Outcome_OS) ~ lp_D,
                            data = boot_data, timewt = "n/G2")

      return(conc_D$concordance - conc_C$concordance)
    }, error = function(e) {
      return(NA)
    })
  }

  set.seed(SEED)
  boot_result <- boot(data = df, statistic = boot_delta_c, R = N_BOOTSTRAP)

  # BCa confidence interval
  boot_ci <- tryCatch(
    boot.ci(boot_result, type = "bca", conf = 1 - ALPHA),
    error = function(e) {
      cat("  BCa CI failed, using percentile method.\n")
      boot.ci(boot_result, type = "perc", conf = 1 - ALPHA)
    }
  )

  if (!is.null(boot_ci$bca)) {
    delta_c_ci <- boot_ci$bca[4:5]
  } else if (!is.null(boot_ci$percent)) {
    delta_c_ci <- boot_ci$percent[4:5]
  } else {
    delta_c_ci <- quantile(boot_result$t, c(0.025, 0.975), na.rm = TRUE)
  }

  cat("\n═══ PRIMARY ESTIMAND RESULT ═══\n")
  cat("ΔC (Model D - Model C):", round(delta_c, 4), "\n")
  cat("95% Bootstrap CI: [", round(delta_c_ci[1], 4), ",",
      round(delta_c_ci[2], 4), "]\n")
  cat("Pre-specified threshold (ΔC ≥ 0.05):",
      ifelse(delta_c >= DELTA_C_THRESHOLD, "MET", "NOT MET"), "\n")
  cat("CI excludes zero:",
      ifelse(delta_c_ci[1] > 0, "YES (Model D superior)",
             ifelse(delta_c_ci[2] < 0, "YES (Model C superior)", "NO")), "\n")

  # Save primary result
  primary_result <- data.frame(
    Comparison = "Model D vs Model C (Layer 3)",
    Delta_C = round(delta_c, 4),
    CI_lower = round(delta_c_ci[1], 4),
    CI_upper = round(delta_c_ci[2], 4),
    Threshold_met = delta_c >= DELTA_C_THRESHOLD,
    Conclusion = ifelse(delta_c >= DELTA_C_THRESHOLD & delta_c_ci[1] > 0,
                        "LNR significantly superior",
                        ifelse(delta_c_ci[2] < 0,
                               "N-stage superior",
                               "No significant difference"))
  )
  save_table(primary_result, "PRIMARY_delta_C_result")

  # Also compute ΔC for Layer 1 and Layer 2 to show the trend
  if (!is.null(models$A) && !is.null(models$B)) {
    c_A <- c_results$C_statistic[c_results$Model == "A"]
    c_B <- c_results$C_statistic[c_results$Model == "B"]
    cat("\nLayer 1 ΔC (B - A):", round(c_B - c_A, 4), "\n")
  }
  if (!is.null(models$C2) && !is.null(models$D2)) {
    c_C2 <- c_results$C_statistic[c_results$Model == "C2"]
    c_D2 <- c_results$C_statistic[c_results$Model == "D2"]
    cat("Layer 2 ΔC (D2 - C2):", round(c_D2 - c_C2, 4), "\n")
  }

} else {
  cat("  Models C and D not available (tier too low for MVA).\n")
  cat("  Using Layer 1 comparison: Model B vs Model A.\n")

  if (!is.null(models$A) && !is.null(models$B)) {
    c_A <- c_results$C_statistic[c_results$Model == "A"]
    c_B <- c_results$C_statistic[c_results$Model == "B"]
    delta_c <- c_B - c_A
    cat("ΔC (Model B - Model A):", round(delta_c, 4), "\n")
    cat("NOTE: This is a hypothesis-generating comparison (inadequate tier).\n")
  }
}

# ═══════════════════════════════════════════════════════════════════════════════
# 3. DECISION-CURVE ANALYSIS (Co-Primary)
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_dca) {
  section_header("DECISION-CURVE ANALYSIS (Co-Primary)")

  # DCA requires predicted probabilities of the event
  # Use 5-year survival prediction as the basis

  # Predicted probabilities from coxph models
  # Using 5-year risk = 1 - survival probability at 5 years

  tryCatch({
    # Refit with coxph for easier prediction
    if (ANALYTIC_TIER$permit_rcs) {
      cox_C_dca <- coxph(Surv(Time_survival, Outcome_OS) ~
                           N_stage + rcs(LN_total, 3) + AG + Sex_f +
                           T_stage_group + Grade_f + LVI_f,
                         data = df)
      cox_D_dca <- coxph(Surv(Time_survival, Outcome_OS) ~
                           rcs(LNR_prop, 3) + rcs(LN_total, 3) + AG + Sex_f +
                           T_stage_group + Grade_f + LVI_f,
                         data = df)
    } else {
      cox_C_dca <- coxph(Surv(Time_survival, Outcome_OS) ~
                           N_stage + LN_total + AG + Sex_f +
                           T_stage_group + Grade_f + LVI_f,
                         data = df)
      cox_D_dca <- coxph(Surv(Time_survival, Outcome_OS) ~
                           LNR_prop + LN_total + AG + Sex_f +
                           T_stage_group + Grade_f + LVI_f,
                         data = df)
    }

    # Baseline survival at 5 years (1826 days)
    t_star <- RMST_TAU_DAYS
    baseline_C <- basehaz(cox_C_dca, centered = FALSE)
    baseline_D <- basehaz(cox_D_dca, centered = FALSE)

    # 5-year predicted risk for each patient
    H0_C <- approx(baseline_C$time, baseline_C$hazard, xout = t_star, rule = 2)$y
    H0_D <- approx(baseline_D$time, baseline_D$hazard, xout = t_star, rule = 2)$y

    lp_C <- predict(cox_C_dca, type = "lp")
    lp_D <- predict(cox_D_dca, type = "lp")

    df$risk_C <- 1 - exp(-H0_C * exp(lp_C))
    df$risk_D <- 1 - exp(-H0_D * exp(lp_D))

    # Create binary outcome for 5-year event
    df$event_5yr <- ifelse(df$Outcome_OS == 1 & df$Time_survival <= t_star, 1, 0)

    # Run DCA
    dca_result <- dca(
      Surv(Time_survival, Outcome_OS) ~ risk_C + risk_D,
      data = df,
      time = t_star,
      thresholds = DCA_THRESHOLDS
    )

    # DCA plot
    p_dca <- plot(dca_result) +
      labs(title = "Decision Curve Analysis: Model C vs Model D",
           subtitle = paste0("5-year overall survival | Threshold range: ",
                             DCA_THRESHOLD_MIN * 100, "%-",
                             DCA_THRESHOLD_MAX * 100, "%")) +
      theme_publication()
    save_figure(p_dca, "Fig_DCA_ModelC_vs_ModelD")

    cat("DCA plot saved. Visual interpretation:\n")
    cat("  - Model with higher net benefit across threshold range is preferred.\n")
    cat("  - If curves are close, neither model has clear clinical advantage.\n")

  }, error = function(e) {
    cat("DCA computation encountered an error:", e$message, "\n")
    cat("This may occur with small sample sizes or sparse events.\n")
  })

} else {
  cat("\n  DCA not permitted at Tier", ANALYTIC_TIER$tier_code, "\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 4. CALIBRATION (Primary Secondary Endpoint)
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_mva && !is.null(models$D)) {
  section_header("CALIBRATION ASSESSMENT")

  for (mname in c("C", "D")) {
    if (!is.null(models[[mname]])) {
      cat(sprintf("\n━━━ Calibration: Model %s ━━━\n", mname))

      tryCatch({
        cal <- calibrate(models[[mname]], B = min(N_BOOTSTRAP, 200),
                         u = RMST_TAU_DAYS, cmethod = "KM")

        pdf(file.path(DIR_FIGURES, paste0("Fig_calibration_Model_", mname, ".pdf")),
            width = FIG_WIDTH, height = FIG_HEIGHT)
        plot(cal, main = paste("Calibration Plot — Model", mname,
                               "(5-year OS)"),
             xlab = "Predicted 5-Year Survival",
             ylab = "Observed 5-Year Survival",
             subtitles = TRUE)
        abline(0, 1, lty = 2, col = "grey50")
        dev.off()

        cat("  Calibration plot saved.\n")
      }, error = function(e) {
        cat("  Calibration failed:", e$message, "\n")
        cat("  This may be due to insufficient events or extreme predictions.\n")
      })
    }
  }
}

# ═══════════════════════════════════════════════════════════════════════════════
# 5. BOOTSTRAP OPTIMISM-CORRECTED C-STATISTIC (Internal Validation)
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_mva) {
  section_header("BOOTSTRAP INTERNAL VALIDATION")

  for (mname in c("C", "D")) {
    if (!is.null(models[[mname]])) {
      cat(sprintf("\n━━━ Bootstrap validation: Model %s ━━━\n", mname))

      tryCatch({
        val <- validate(models[[mname]], B = N_BOOTSTRAP, method = "boot")
        print(val)

        # Extract optimism-corrected Dxy → C
        dxy_corrected <- val["Dxy", "index.corrected"]
        c_corrected <- 0.5 * (dxy_corrected + 1)
        cat(sprintf("  Optimism-corrected C-statistic: %.4f\n", c_corrected))
        cat(sprintf("  Optimism: %.4f\n", val["Dxy", "optimism"] / 2))

      }, error = function(e) {
        cat("  Bootstrap validation failed:", e$message, "\n")
      })
    }
  }
}

# ═══════════════════════════════════════════════════════════════════════════════
# 6. AIC / BIC COMPARISON
# ═══════════════════════════════════════════════════════════════════════════════

section_header("AIC / BIC COMPARISON")

aic_bic <- model_summary[, c("Model", "Layer", "AIC", "BIC")]
cat("AIC / BIC comparison across models:\n")
print(aic_bic)

# Identify best model by AIC
best_aic <- aic_bic$Model[which.min(aic_bic$AIC)]
best_bic <- aic_bic$Model[which.min(aic_bic$BIC)]
cat("\nBest model by AIC:", best_aic, "\n")
cat("Best model by BIC:", best_bic, "\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. EXPLORATORY: NRI AND IDI (Appendix Only)
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_mva && !is.null(models$C) && !is.null(models$D)) {
  section_header("EXPLORATORY: NRI AND IDI (APPENDIX ONLY)")
  cat("NOTE: These metrics are reported in supplementary appendix only.\n")
  cat("They are NOT used for primary conclusions per protocol.\n\n")

  tryCatch({
    # Predicted probabilities (linear predictors)
    lp_model_C <- predict(models$C, type = "lp")
    lp_model_D <- predict(models$D, type = "lp")

    # Continuous NRI using survIDINRI or nricens package
    # Simple implementation: direction of reclassification
    improved <- (lp_model_D > lp_model_C & df$Outcome_OS == 1) |
                (lp_model_D < lp_model_C & df$Outcome_OS == 0)
    worsened <- (lp_model_D < lp_model_C & df$Outcome_OS == 1) |
                (lp_model_D > lp_model_C & df$Outcome_OS == 0)

    nri_events <- mean(lp_model_D[df$Outcome_OS == 1] >
                        lp_model_C[df$Outcome_OS == 1]) -
                  mean(lp_model_D[df$Outcome_OS == 1] <
                        lp_model_C[df$Outcome_OS == 1])
    nri_nonevents <- mean(lp_model_D[df$Outcome_OS == 0] <
                           lp_model_C[df$Outcome_OS == 0]) -
                     mean(lp_model_D[df$Outcome_OS == 0] >
                           lp_model_C[df$Outcome_OS == 0])
    nri_continuous <- nri_events + nri_nonevents

    cat("Continuous NRI (Pencina 2011):\n")
    cat("  NRI_events:", round(nri_events, 4), "\n")
    cat("  NRI_nonevents:", round(nri_nonevents, 4), "\n")
    cat("  NRI_total:", round(nri_continuous, 4), "\n\n")

    # IDI
    idi <- mean(lp_model_D[df$Outcome_OS == 1]) -
           mean(lp_model_D[df$Outcome_OS == 0]) -
           (mean(lp_model_C[df$Outcome_OS == 1]) -
            mean(lp_model_C[df$Outcome_OS == 0]))
    cat("IDI:", round(idi, 4), "\n")

    nri_idi_result <- data.frame(
      Metric = c("NRI_events", "NRI_nonevents", "NRI_total", "IDI"),
      Value = round(c(nri_events, nri_nonevents, nri_continuous, idi), 4),
      Note = rep("Exploratory — appendix only", 4)
    )
    save_table(nri_idi_result, "appendix_NRI_IDI")

  }, error = function(e) {
    cat("NRI/IDI computation failed:", e$message, "\n")
  })
}

# ═══════════════════════════════════════════════════════════════════════════════
# 8. C-STATISTIC COMPARISON PLOT
# ═══════════════════════════════════════════════════════════════════════════════

section_header("C-STATISTIC COMPARISON PLOT")

if (nrow(c_results) > 0) {
  c_results$Model_label <- paste0("Model ", c_results$Model)
  c_results$Model_label <- factor(c_results$Model_label,
                                   levels = rev(c_results$Model_label))

  # Assign layers for colouring
  c_results$Layer <- ifelse(c_results$Model %in% c("A", "B"), "Layer 1",
                     ifelse(c_results$Model %in% c("C2", "D2"), "Layer 2",
                            "Layer 3"))

  p_cstat <- ggplot(c_results, aes(x = C_statistic, y = Model_label,
                                    colour = Layer)) +
    geom_point(size = 4) +
    geom_errorbarh(aes(xmin = CI_lower, xmax = CI_upper), height = 0.3) +
    geom_vline(xintercept = 0.5, linetype = "dotted", colour = "grey60") +
    scale_colour_manual(values = c("Layer 1" = "#4393C3",
                                    "Layer 2" = "#2166AC",
                                    "Layer 3" = "#053061")) +
    labs(title = "C-Statistic Comparison Across Model Layers",
         subtitle = "Uno's C-statistic with 95% CI",
         x = "C-statistic",
         y = NULL,
         colour = "Model Layer") +
    theme_publication() +
    theme(legend.position = "right")
  save_figure(p_cstat, "Fig_C_statistic_comparison")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 9. SAVE RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

save(c_results, model_summary,
     file = file.path(DIR_MODELS, "phase3_performance_metrics.RData"))

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 05 COMPLETE\n")
cat(strrep("=", 78), "\n")



  SCRIPT 05: PERFORMANCE METRICS (Phase 3)

Analytic tier: Inadequate (< 80 events) 


  UNO'S C-STATISTIC

Uno C-stat truncation (75th percentile of event times): 895 days

  Model A: C = 0.4248 (95% CI: 0.3418 - 0.5078)
  Model B: C = 0.4667 (95% CI: 0.3637 - 0.5698)

C-statistic summary table:
  Model C_statistic     SE CI_lower CI_upper
1     A      0.4248 0.0423   0.3418   0.5078
2     B      0.4667 0.0526   0.3637   0.5698
Saved: C_statistic_all_models .csv

  PRIMARY ΔC COMPARISON (MODEL D vs MODEL C)

  Models C and D not available (tier too low for MVA).
  Using Layer 1 comparison: Model B vs Model A.
ΔC (Model B - Model A): 0.0419 
NOTE: This is a hypothesis-generating comparison (inadequate tier).

  DCA not permitted at Tier 1 

  AIC / BIC COMPARISON

AIC / BIC comparison across models:
      Model          Layer      AIC      BIC
d.f.      A 1 (Unadjusted) 249.4176 251.5448
d.f.1     B 1 (Unadjusted) 252.6141 254.7413

Best model by AIC: A 
Best model by BIC: A 

  C-STA

Warning message:
“`geom_errorbarh()` was deprecated in ggplot2 4.0.0.
ℹ Please use the `orientation` argument of `geom_errorbar()` instead.”
`height` was translated to `width`.
`height` was translated to `width`.


Saved: Fig_C_statistic_comparison (PDF + PNG)

  SCRIPT 05 COMPLETE


## 6

In [ ]:
################################################################################
#  06_reclassification.R — Phase 4: Reclassification Analysis
#  Protocol Section 7.5 | SO4
#
#  Cross-tabulation of N-stage × LNR tier, reclassification proportion,
#  and survival validation of reclassified patients
################################################################################

load(file.path(DIR_DATA, "analytic_dataset.RData"))

section_header("SCRIPT 06: RECLASSIFICATION ANALYSIS (Phase 4)")

cat("Analytic cohort: N =", nrow(df), "| OS events:",
    sum(df$Outcome_OS == 1), "\n\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CROSS-TABULATION: N-STAGE × LNR TIER
# ═══════════════════════════════════════════════════════════════════════════════

section_header("N-STAGE × LNR TIER CROSS-TABULATION")

cross_tab <- table(df$N_stage, df$LNR_tier)
cat("Counts:\n")
print(addmargins(cross_tab))

cat("\nRow percentages (within each N-stage):\n")
row_pct <- round(prop.table(cross_tab, margin = 1) * 100, 1)
print(row_pct)

cat("\nColumn percentages (within each LNR tier):\n")
col_pct <- round(prop.table(cross_tab, margin = 2) * 100, 1)
print(col_pct)

# ═══════════════════════════════════════════════════════════════════════════════
# 2. RECLASSIFICATION ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("RECLASSIFICATION ANALYSIS")

# Define "concordant" vs "discordant" classification
# N1 patients → LOW/INTERMEDIATE LNR = concordant (expected lower risk)
#             → HIGH LNR = reclassified upward (higher risk than N-stage implies)
# N2 patients → HIGH LNR = concordant (expected higher risk)
#             → LOW/INTERMEDIATE LNR = reclassified downward (lower risk)

df$reclass_group <- NA_character_

# N1 patients
n1_mask <- df$N_stage == "N1"
df$reclass_group[n1_mask & df$LNR_tier == "Low (<0.05)"] <- "N1_LNR-Low (concordant)"
df$reclass_group[n1_mask & df$LNR_tier == "Intermediate (0.05-0.20)"] <- "N1_LNR-Intermediate"
df$reclass_group[n1_mask & df$LNR_tier == "High (>0.20)"] <- "N1_LNR-High (upclassified)"

# N2 patients
n2_mask <- df$N_stage == "N2"
df$reclass_group[n2_mask & df$LNR_tier == "Low (<0.05)"] <- "N2_LNR-Low (downclassified)"
df$reclass_group[n2_mask & df$LNR_tier == "Intermediate (0.05-0.20)"] <- "N2_LNR-Intermediate (downclassified)"
df$reclass_group[n2_mask & df$LNR_tier == "High (>0.20)"] <- "N2_LNR-High (concordant)"

cat("Reclassification groups:\n")
print(table(df$reclass_group, useNA = "ifany"))

# Proportions
n_total <- nrow(df)
n_concordant <- sum(grepl("concordant", df$reclass_group, ignore.case = TRUE),
                    na.rm = TRUE)
n_reclassified <- n_total - n_concordant

cat("\nConcordant (N-stage and LNR agree on risk level):", n_concordant,
    "(", round(100 * n_concordant / n_total, 1), "%)\n")
cat("Reclassified (N-stage and LNR disagree):", n_reclassified,
    "(", round(100 * n_reclassified / n_total, 1), "%)\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. SURVIVAL BY RECLASSIFICATION CELL
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SURVIVAL BY RECLASSIFICATION CELL")

# Create combined cell variable
df$reclass_cell <- paste0(df$N_stage, " / ", df$LNR_tier)
df$reclass_cell <- factor(df$reclass_cell)

cat("Cell counts and events:\n")
cell_summary <- df %>%
  group_by(reclass_cell) %>%
  summarise(
    N = n(),
    Events_OS = sum(Outcome_OS == 1),
    Median_surv_days = median(Time_survival),
    .groups = "drop"
  ) %>%
  as.data.frame()
print(cell_summary)
save_table(cell_summary, "reclassification_cell_summary")

# KM curves by reclassification cell (only for cells with sufficient data)
cells_with_data <- cell_summary$reclass_cell[cell_summary$N >= 3 &
                                               cell_summary$Events_OS >= 1]

if (length(cells_with_data) >= 2) {
  df_km_reclass <- df[df$reclass_cell %in% cells_with_data, ]

  km_reclass <- survfit(Surv(Time_survival, Outcome_OS) ~ reclass_cell,
                         data = df_km_reclass)

  # Custom colours
  n_cells <- length(cells_with_data)
  cell_colours <- viridis(n_cells)

  p_km_reclass <- ggsurvplot(
    km_reclass, data = df_km_reclass,
    palette = cell_colours,
    risk.table = TRUE,
    risk.table.height = 0.30,
    pval = TRUE,
    conf.int = FALSE,
    xlab = "Time (days)",
    ylab = "Overall Survival Probability",
    title = "Overall Survival by N-Stage × LNR Tier Classification",
    subtitle = "Each cell represents a unique concordance/discordance group",
    legend.title = "N-Stage / LNR Tier",
    ggtheme = theme_publication(),
    break.time.by = 365.25,
    legend = "right"
  )

  pdf(file.path(DIR_FIGURES, "Fig_KM_reclassification_cells.pdf"),
      width = FIG_WIDTH + 2, height = FIG_HEIGHT + 2)
  print(p_km_reclass)
  dev.off()
  png(file.path(DIR_FIGURES, "Fig_KM_reclassification_cells.png"),
      width = FIG_WIDTH + 2, height = FIG_HEIGHT + 2, units = "in", res = FIG_DPI)
  print(p_km_reclass)
  dev.off()
  cat("Saved: Fig_KM_reclassification_cells\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 4. RMST BY RECLASSIFICATION CELL
# ═══════════════════════════════════════════════════════════════════════════════

section_header("RMST BY RECLASSIFICATION CELL")

tau <- min(RMST_TAU_DAYS, max(df$Time_survival, na.rm = TRUE))

rmst_cells <- data.frame()
for (cell in levels(df$reclass_cell)) {
  sub <- df[df$reclass_cell == cell, ]
  if (nrow(sub) >= 2 && sum(sub$Outcome_OS == 1) >= 1) {
    km_sub <- survfit(Surv(Time_survival, Outcome_OS) ~ 1, data = sub)
    summ <- summary(km_sub, rmean = tau)$table
    # Use grep to find rmean fields (names vary across survival pkg versions)
    rmean_idx <- grep("rmean", names(summ))[1]
    se_idx    <- grep("se.*rmean", names(summ))[1]
    rmst_cells <- rbind(rmst_cells, data.frame(
      Cell = cell,
      N = nrow(sub),
      Events = sum(sub$Outcome_OS == 1),
      RMST_days = round(unname(summ[rmean_idx]), 0),
      RMST_years = round(unname(summ[rmean_idx]) / 365.25, 2),
      SE = round(unname(summ[se_idx]), 0),
      stringsAsFactors = FALSE
    ))
  }
}

cat("RMST at", round(tau / 365.25, 1), "years by reclassification cell:\n")
print(rmst_cells)
save_table(rmst_cells, "RMST_reclassification_cells")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. MOSAIC/HEATMAP: RECLASSIFICATION WITH SURVIVAL
# ═══════════════════════════════════════════════════════════════════════════════

section_header("RECLASSIFICATION HEATMAP")

# Create a heatmap showing cell counts with survival overlay
cross_df <- as.data.frame(cross_tab)
names(cross_df) <- c("N_Stage", "LNR_Tier", "Count")

# Calculate survival rate per cell
cross_df$Mortality_rate <- NA
for (i in seq_len(nrow(cross_df))) {
  sub <- df[df$N_stage == cross_df$N_Stage[i] &
             df$LNR_tier == cross_df$LNR_Tier[i], ]
  if (nrow(sub) > 0) {
    cross_df$Mortality_rate[i] <- round(100 * sum(sub$Outcome_OS == 1) / nrow(sub), 1)
  }
}

p_heatmap <- ggplot(cross_df, aes(x = LNR_Tier, y = N_Stage, fill = Mortality_rate)) +
  geom_tile(colour = "white", linewidth = 1.5) +
  geom_text(aes(label = paste0("N = ", Count, "\n",
                                ifelse(is.na(Mortality_rate), "",
                                       paste0(Mortality_rate, "% died")))),
            colour = "white", fontface = "bold", size = 4) +
  scale_fill_gradient(low = "#2166AC", high = "#B2182B", na.value = "grey80",
                      name = "Mortality\nRate (%)") +
  labs(title = "Reclassification Heatmap: N-Stage × LNR Tier",
       subtitle = "Cell size and mortality rate for OS",
       x = "LNR Tier (Rosenberg)", y = "N-Stage") +
  theme_publication() +
  theme(axis.text.x = element_text(angle = 15, hjust = 1))
save_figure(p_heatmap, "Fig_reclassification_heatmap")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# Full reclassification table for manuscript
# Create matching composite key for merge
cross_df$Cell <- paste0(cross_df$N_Stage, " / ", cross_df$LNR_Tier)
reclass_table <- merge(cross_df, rmst_cells, by = "Cell", all.x = TRUE)
cross_df$Cell <- NULL   # remove temp key
save_table(cross_df, "reclassification_table_full")

save(cross_tab, cell_summary, rmst_cells,
     file = file.path(DIR_MODELS, "phase4_reclassification.RData"))

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 06 COMPLETE\n")
cat(strrep("=", 78), "\n")



  SCRIPT 06: RECLASSIFICATION ANALYSIS (Phase 4)

Analytic cohort: N = 62 | OS events: 34 


  N-STAGE × LNR TIER CROSS-TABULATION

Counts:
     
      Low (<0.05) Intermediate (0.05-0.20) High (>0.20) Sum
  N1            7                       27            7  41
  N2            0                        2           19  21
  Sum           7                       29           26  62

Row percentages (within each N-stage):
    
     Low (<0.05) Intermediate (0.05-0.20) High (>0.20)
  N1        17.1                     65.9         17.1
  N2         0.0                      9.5         90.5

Column percentages (within each LNR tier):
    
     Low (<0.05) Intermediate (0.05-0.20) High (>0.20)
  N1       100.0                     93.1         26.9
  N2         0.0                      6.9         73.1

  RECLASSIFICATION ANALYSIS

Reclassification groups:

          N1_LNR-High (upclassified)                  N1_LNR-Intermediate 
                                   7                      

Ignoring unknown labels:
• colour : "N-Stage / LNR Tier"
Ignoring unknown labels:
• colour : "N-Stage / LNR Tier"


Saved: Fig_KM_reclassification_cells

  RMST BY RECLASSIFICATION CELL

RMST at 5 years by reclassification cell:
                           Cell  N Events RMST_days RMST_years  SE
1             N1 / High (>0.20)  7      2      1491       4.08 208
2 N1 / Intermediate (0.05-0.20) 27     13      1161       3.18 143
3              N1 / Low (<0.05)  7      4      1222       3.35 230
4             N2 / High (>0.20) 19     13       905       2.48 173
5 N2 / Intermediate (0.05-0.20)  2      2       793       2.17 415
Saved: RMST_reclassification_cells .csv

  RECLASSIFICATION HEATMAP

Saved: Fig_reclassification_heatmap (PDF + PNG)
Saved: reclassification_table_full .csv

  SCRIPT 06 COMPLETE


## 7

In [ ]:
################################################################################
#  07_subgroup_sensitivity.R — Phase 5: Subgroup & Sensitivity Analyses
#  Protocol Section 7.6 | SO5
#
#  Pre-specified subgroups: TLE <12 vs >=12, T3/T4 vs T1/T2, N1 vs N2
#  Pre-specified sensitivities: landmark, binary TLE, alt LNR cutpoints
################################################################################

load(file.path(DIR_DATA, "analytic_dataset.RData"))
load(file.path(DIR_MODELS, "phase3_cox_models.RData"))

section_header("SCRIPT 07: SUBGROUP & SENSITIVITY ANALYSES (Phase 5)")

cat("Analytic cohort: N =", nrow(df), "| OS events:",
    sum(df$Outcome_OS == 1), "\n")
cat("Analytic tier:", ANALYTIC_TIER$label, "\n\n")

# Re-establish rms datadist
dd <- datadist(df)
options(datadist = "dd")

# Helper to compute C-stat for a model within a subgroup
compute_c_subgroup <- function(formula, data, label) {
  tryCatch({
    fit <- coxph(formula, data = data)
    lp <- predict(fit, type = "lp")
    conc <- concordance(Surv(Time_survival, Outcome_OS) ~ lp,
                        data = data, timewt = "n/G2")
    data.frame(
      Subgroup = label,
      N = nrow(data),
      Events = sum(data$Outcome_OS == 1),
      C_statistic = round(conc$concordance, 4),
      SE = round(sqrt(conc$var), 4),
      stringsAsFactors = FALSE
    )
  }, error = function(e) {
    data.frame(
      Subgroup = label, N = nrow(data),
      Events = sum(data$Outcome_OS == 1),
      C_statistic = NA, SE = NA, stringsAsFactors = FALSE
    )
  })
}

# ═══════════════════════════════════════════════════════════════════════════════
# 1. SUBGROUP ANALYSIS: TLE < 12 vs >= 12 (H4 — Primary subgroup)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SUBGROUP: TLE ADEQUACY (< 12 vs >= 12)")

cat("This is the primary subgroup of mechanistic interest (H4).\n")
cat("Tests whether LNR advantage is amplified in sub-optimal harvest.\n\n")

# Define formulas based on tier
if (ANALYTIC_TIER$permit_rcs) {
  formula_nstage <- Surv(Time_survival, Outcome_OS) ~ N_stage
  formula_lnr <- Surv(Time_survival, Outcome_OS) ~ LNR_prop
} else {
  formula_nstage <- Surv(Time_survival, Outcome_OS) ~ N_stage
  formula_lnr <- Surv(Time_survival, Outcome_OS) ~ LNR_prop
}

subgroup_results_tle <- data.frame()

for (tle_level in c("Inadequate (<12)", "Adequate (>=12)")) {
  sub_data <- df[df$TLE_adequate == tle_level, ]
  cat(sprintf("\n━━━ TLE %s (N = %d, events = %d) ━━━\n",
              tle_level, nrow(sub_data), sum(sub_data$Outcome_OS == 1)))

  if (nrow(sub_data) >= 10 && sum(sub_data$Outcome_OS == 1) >= 3) {
    # N-stage model C-stat
    c_ns <- compute_c_subgroup(formula_nstage, sub_data,
                                paste("N-stage |", tle_level))
    # LNR model C-stat
    c_lnr <- compute_c_subgroup(formula_lnr, sub_data,
                                 paste("LNR |", tle_level))

    subgroup_results_tle <- rbind(subgroup_results_tle, c_ns, c_lnr)

    delta_c <- c_lnr$C_statistic - c_ns$C_statistic
    cat(sprintf("  N-stage C = %.4f | LNR C = %.4f | ΔC = %.4f\n",
                c_ns$C_statistic, c_lnr$C_statistic, delta_c))

    # KM curves within subgroup
    if (sum(sub_data$Outcome_OS == 1) >= 3 &&
        nlevels(droplevels(sub_data$LNR_tier)) >= 2) {
      km_sub <- survfit(Surv(Time_survival, Outcome_OS) ~ LNR_tier, data = sub_data)
      p_km_sub <- ggsurvplot(
        km_sub, data = sub_data,
        palette = unname(COL_LNR_TIERS),
        pval = TRUE, conf.int = FALSE,
        xlab = "Time (days)", ylab = "Overall Survival",
        title = paste("OS by LNR Tier — TLE", tle_level),
        ggtheme = theme_publication(), break.time.by = 365.25,
        risk.table = TRUE, risk.table.height = 0.25
      )
      pdf(file.path(DIR_FIGURES,
                     paste0("Fig_KM_LNR_TLE_", gsub("[^a-zA-Z0-9]", "", tle_level), ".pdf")),
          width = FIG_WIDTH, height = FIG_HEIGHT + 1.5)
      print(p_km_sub)
      dev.off()
    }
  } else {
    cat("  Insufficient data for subgroup analysis.\n")
  }
}

cat("\nSubgroup comparison summary (TLE adequacy):\n")
print(subgroup_results_tle)
save_table(subgroup_results_tle, "subgroup_TLE_adequacy")

# Formal interaction test (only at tier 4)
if (ANALYTIC_TIER$permit_interaction) {
  cat("\n━━━ Formal Interaction Test: LNR × TLE adequacy ━━━\n")
  int_model <- coxph(Surv(Time_survival, Outcome_OS) ~
                       LNR_prop * TLE_adequate_binary, data = df)
  print(summary(int_model))
  int_anova <- anova(int_model)
  print(int_anova)
} else {
  cat("\nFormal interaction test NOT permitted (requires >= 200 events).\n")
  cat("Subgroup results above are DESCRIPTIVE and EXPLORATORY only.\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 2. SUBGROUP: T-STAGE (T3/T4 vs T1/T2)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SUBGROUP: T-STAGE (T3/T4 vs T1/T2)")

df$T_high <- ifelse(df$ST_main >= 3, "T3/T4", "T1/T2")

subgroup_results_tstage <- data.frame()
for (t_level in c("T1/T2", "T3/T4")) {
  sub_data <- df[df$T_high == t_level, ]
  cat(sprintf("\n━━━ %s (N = %d, events = %d) ━━━\n",
              t_level, nrow(sub_data), sum(sub_data$Outcome_OS == 1)))

  if (nrow(sub_data) >= 10 && sum(sub_data$Outcome_OS == 1) >= 3) {
    c_ns <- compute_c_subgroup(formula_nstage, sub_data, paste("N-stage |", t_level))
    c_lnr <- compute_c_subgroup(formula_lnr, sub_data, paste("LNR |", t_level))
    subgroup_results_tstage <- rbind(subgroup_results_tstage, c_ns, c_lnr)
    delta_c <- c_lnr$C_statistic - c_ns$C_statistic
    cat(sprintf("  N-stage C = %.4f | LNR C = %.4f | ΔC = %.4f\n",
                c_ns$C_statistic, c_lnr$C_statistic, delta_c))
  } else {
    cat("  Insufficient data.\n")
  }
}

print(subgroup_results_tstage)
save_table(subgroup_results_tstage, "subgroup_T_stage")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. SUBGROUP: N1 vs N2
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SUBGROUP: N1 vs N2")

subgroup_results_nstage <- data.frame()
for (n_level in c("N1", "N2")) {
  sub_data <- df[df$N_stage == n_level, ]
  cat(sprintf("\n━━━ %s (N = %d, events = %d) ━━━\n",
              n_level, nrow(sub_data), sum(sub_data$Outcome_OS == 1)))

  if (nrow(sub_data) >= 10 && sum(sub_data$Outcome_OS == 1) >= 3) {
    c_lnr <- compute_c_subgroup(formula_lnr, sub_data, paste("LNR |", n_level))
    subgroup_results_nstage <- rbind(subgroup_results_nstage, c_lnr)
    cat(sprintf("  LNR C-statistic within %s stratum: %.4f\n",
                n_level, c_lnr$C_statistic))
  } else {
    cat("  Insufficient data.\n")
  }
}

print(subgroup_results_nstage)
save_table(subgroup_results_nstage, "subgroup_N_stage")

# ═══════════════════════════════════════════════════════════════════════════════
# 4. SENSITIVITY ANALYSIS: LANDMARK AT 6 MONTHS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SENSITIVITY: 6-MONTH LANDMARK ANALYSIS")

cat("Excluding events and observations within first", LANDMARK_DAYS, "days.\n")
cat("Purpose: Remove early surgical mortality from prognostic signal.\n\n")

df_landmark <- df[df$Time_survival > LANDMARK_DAYS, ]
df_landmark$Time_landmark <- df_landmark$Time_survival - LANDMARK_DAYS

cat("After landmark exclusion: N =", nrow(df_landmark),
    "| Events:", sum(df_landmark$Outcome_OS == 1), "\n")
cat("Excluded:", nrow(df) - nrow(df_landmark), "patients with",
    sum(df$Outcome_OS == 1 & df$Time_survival <= LANDMARK_DAYS),
    "early deaths.\n\n")

if (nrow(df_landmark) >= 20 && sum(df_landmark$Outcome_OS == 1) >= 5) {
  # Refit univariate models on landmark cohort
  cox_ns_lm <- coxph(Surv(Time_landmark, Outcome_OS) ~ N_stage, data = df_landmark)
  cox_lnr_lm <- coxph(Surv(Time_landmark, Outcome_OS) ~ LNR_prop, data = df_landmark)

  cat("N-stage (landmark):\n")
  print(summary(cox_ns_lm))
  cat("\nLNR continuous (landmark):\n")
  print(summary(cox_lnr_lm))

  # C-statistics
  c_ns_lm <- compute_c_subgroup(
    Surv(Time_landmark, Outcome_OS) ~ N_stage, df_landmark, "N-stage (landmark)")
  c_lnr_lm <- compute_c_subgroup(
    Surv(Time_landmark, Outcome_OS) ~ LNR_prop, df_landmark, "LNR (landmark)")

  landmark_results <- rbind(c_ns_lm, c_lnr_lm)
  print(landmark_results)
  save_table(landmark_results, "sensitivity_landmark_6mo")

  # KM by LNR tier (landmark)
  km_lnr_lm <- survfit(Surv(Time_landmark, Outcome_OS) ~ LNR_tier,
                         data = df_landmark)
  p_km_lm <- ggsurvplot(
    km_lnr_lm, data = df_landmark, palette = unname(COL_LNR_TIERS),
    pval = TRUE, conf.int = FALSE, risk.table = TRUE, risk.table.height = 0.25,
    xlab = "Time from Landmark (days)", ylab = "Overall Survival",
    title = "OS by LNR Tier — 6-Month Landmark Analysis",
    ggtheme = theme_publication(), break.time.by = 365.25
  )
  pdf(file.path(DIR_FIGURES, "Fig_KM_landmark_6mo.pdf"),
      width = FIG_WIDTH, height = FIG_HEIGHT + 1.5)
  print(p_km_lm)
  dev.off()
} else {
  cat("  Insufficient data after landmark exclusion.\n")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 5. SENSITIVITY: BINARY TLE IN MVA
# ═══════════════════════════════════════════════════════════════════════════════

if (ANALYTIC_TIER$permit_mva) {
  section_header("SENSITIVITY: BINARY TLE (< 12 vs >= 12) IN MVA")

  cat("Replacing continuous/RCS TLE with binary (< 12 vs >= 12).\n\n")

  # Model C with binary TLE
  cox_C_binTLE <- coxph(Surv(Time_survival, Outcome_OS) ~
                          N_stage + TLE_adequate_binary + AG + Sex_f +
                          T_stage_group + Grade_f + LVI_f,
                        data = df)
  cat("Model C (binary TLE):\n")
  print(summary(cox_C_binTLE))

  # Model D with binary TLE
  cox_D_binTLE <- coxph(Surv(Time_survival, Outcome_OS) ~
                          LNR_prop + TLE_adequate_binary + AG + Sex_f +
                          T_stage_group + Grade_f + LVI_f,
                        data = df)
  cat("\nModel D (binary TLE):\n")
  print(summary(cox_D_binTLE))

  # C-statistics
  c_C_bin <- compute_c_subgroup(
    Surv(Time_survival, Outcome_OS) ~ N_stage + TLE_adequate_binary + AG + Sex_f +
      T_stage_group + Grade_f + LVI_f,
    df, "Model C (binary TLE)")
  c_D_bin <- compute_c_subgroup(
    Surv(Time_survival, Outcome_OS) ~ LNR_prop + TLE_adequate_binary + AG + Sex_f +
      T_stage_group + Grade_f + LVI_f,
    df, "Model D (binary TLE)")

  binTLE_results <- rbind(c_C_bin, c_D_bin)
  binTLE_results$Delta_C <- c(NA, c_D_bin$C_statistic - c_C_bin$C_statistic)
  print(binTLE_results)
  save_table(binTLE_results, "sensitivity_binary_TLE")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 6. SENSITIVITY: ALTERNATIVE LNR CUTPOINTS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SENSITIVITY: ALTERNATIVE LNR CUTPOINTS")

# --- 6.1 Cutpoints 0.05/0.40 ---
cat("━━━ LNR tiers with 0.05/0.40 cutpoints ━━━\n")
km_alt1 <- survfit(Surv(Time_survival, Outcome_OS) ~ LNR_tier_alt1, data = df)
cat("Distribution:\n")
print(table(df$LNR_tier_alt1))
lr_alt1 <- survdiff(Surv(Time_survival, Outcome_OS) ~ LNR_tier_alt1, data = df)
cat("Log-rank p:", format_pval(1 - pchisq(lr_alt1$chisq,
                                           df = length(lr_alt1$n) - 1)), "\n\n")

# --- 6.2 Binary cutpoint at 0.20 ---
cat("━━━ LNR binary with 0.20 cutpoint ━━━\n")
km_alt2 <- survfit(Surv(Time_survival, Outcome_OS) ~ LNR_tier_binary, data = df)
cat("Distribution:\n")
print(table(df$LNR_tier_binary))
lr_alt2 <- survdiff(Surv(Time_survival, Outcome_OS) ~ LNR_tier_binary, data = df)
cat("Log-rank p:", format_pval(1 - pchisq(lr_alt2$chisq,
                                           df = length(lr_alt2$n) - 1)), "\n")

# KM plot for binary threshold
p_km_binary <- ggsurvplot(
  km_alt2, data = df,
  palette = c("#2166AC", "#B2182B"),
  risk.table = TRUE, risk.table.height = 0.25,
  pval = TRUE, pval.method = TRUE,
  conf.int = TRUE, conf.int.alpha = 0.15,
  xlab = "Time (days)", ylab = "Overall Survival Probability",
  title = "OS by LNR Binary (0.20 Threshold) — Sensitivity",
  legend.title = "LNR Group",
  ggtheme = theme_publication(), break.time.by = 365.25
)
pdf(file.path(DIR_APPENDIX, "Fig_KM_LNR_binary_020.pdf"),
    width = FIG_WIDTH, height = FIG_HEIGHT + 1.5)
print(p_km_binary)
dev.off()

# --- 6.3 Compare C-statistics across LNR categorisations ---
cat("\n━━━ C-statistic comparison across LNR categorisations ━━━\n")
lnr_cuts_compare <- data.frame()
for (tier_var in c("LNR_tier", "LNR_tier_alt1", "LNR_tier_binary")) {
  fml <- as.formula(paste("Surv(Time_survival, Outcome_OS) ~", tier_var))
  c_res <- compute_c_subgroup(fml, df, tier_var)
  lnr_cuts_compare <- rbind(lnr_cuts_compare, c_res)
}
# Add continuous
c_cont <- compute_c_subgroup(
  Surv(Time_survival, Outcome_OS) ~ LNR_prop, df, "LNR_prop (continuous)")
lnr_cuts_compare <- rbind(lnr_cuts_compare, c_cont)

print(lnr_cuts_compare)
save_table(lnr_cuts_compare, "sensitivity_LNR_cutpoints_comparison")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. COMPILE ALL SUBGROUP/SENSITIVITY RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("SUMMARY TABLE: ALL SUBGROUP & SENSITIVITY ANALYSES")

all_subgroup <- rbind(
  if (nrow(subgroup_results_tle) > 0)
    cbind(Analysis = "TLE adequacy subgroup", subgroup_results_tle) else NULL,
  if (nrow(subgroup_results_tstage) > 0)
    cbind(Analysis = "T-stage subgroup", subgroup_results_tstage) else NULL,
  if (nrow(subgroup_results_nstage) > 0)
    cbind(Analysis = "N-stage subgroup", subgroup_results_nstage) else NULL,
  if (exists("landmark_results"))
    cbind(Analysis = "6-month landmark", landmark_results) else NULL
)

if (!is.null(all_subgroup) && nrow(all_subgroup) > 0) {
  cat("Complete subgroup & sensitivity results:\n")
  print(all_subgroup)
  save_table(all_subgroup, "all_subgroup_sensitivity_results")
}

save(subgroup_results_tle, subgroup_results_tstage, subgroup_results_nstage,
     lnr_cuts_compare,
     file = file.path(DIR_MODELS, "phase5_subgroup_sensitivity.RData"))

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 07 COMPLETE\n")
cat(strrep("=", 78), "\n")



  SCRIPT 07: SUBGROUP & SENSITIVITY ANALYSES (Phase 5)

Analytic cohort: N = 62 | OS events: 34 
Analytic tier: Inadequate (< 80 events) 


  SUBGROUP: TLE ADEQUACY (< 12 vs >= 12)

This is the primary subgroup of mechanistic interest (H4).
Tests whether LNR advantage is amplified in sub-optimal harvest.


━━━ TLE Inadequate (<12) (N = 16, events = 13) ━━━
  N-stage C = 0.4958 | LNR C = 0.3644 | ΔC = -0.1314


Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'OS by LNR Tier — TLE Inadequate (<12)' in 'mbcsToSbcs': - substituted for — (U+2014)”



━━━ TLE Adequate (>=12) (N = 46, events = 21) ━━━
  N-stage C = 0.3875 | LNR C = 0.4412 | ΔC = 0.0537


Ignoring unknown labels:
• colour : "Strata"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'OS by LNR Tier — TLE Adequate (>=12)' in 'mbcsToSbcs': - substituted for — (U+2014)”



Subgroup comparison summary (TLE adequacy):
                    Subgroup  N Events C_statistic     SE
1 N-stage | Inadequate (<12) 16     13      0.4958 0.0776
2     LNR | Inadequate (<12) 16     13      0.3644 0.0829
3  N-stage | Adequate (>=12) 46     21      0.3875 0.0521
4      LNR | Adequate (>=12) 46     21      0.4412 0.0690
Saved: subgroup_TLE_adequacy .csv

Formal interaction test NOT permitted (requires >= 200 events).
Subgroup results above are DESCRIPTIVE and EXPLORATORY only.

  SUBGROUP: T-STAGE (T3/T4 vs T1/T2)


━━━ T1/T2 (N = 9, events = 4) ━━━
  Insufficient data.

━━━ T3/T4 (N = 53, events = 30) ━━━
  N-stage C = 0.4230 | LNR C = 0.4470 | ΔC = 0.0240
         Subgroup  N Events C_statistic     SE
1 N-stage | T3/T4 53     30       0.423 0.0467
2     LNR | T3/T4 53     30       0.447 0.0575
Saved: subgroup_T_stage .csv

  SUBGROUP: N1 vs N2


━━━ N1 (N = 41, events = 19) ━━━
  LNR C-statistic within N1 stratum: 0.4478

━━━ N2 (N = 21, events = 15) ━━━
  LNR C-statisti

...

agg_record_9db76753531d 
                      2


  SENSITIVITY: ALTERNATIVE LNR CUTPOINTS

━━━ LNR tiers with 0.05/0.40 cutpoints ━━━
Distribution:

             Low (<0.05) Intermediate (0.05-0.40)             High (>0.40) 
                       7                       37                       18 
Log-rank p: 0.648 

━━━ LNR binary with 0.20 cutpoint ━━━
Distribution:

  Low (<0.20) High (>=0.20) 
           36            26 
Log-rank p: 0.679 


Ignoring unknown labels:
• colour : "LNR Group"
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“for 'OS by LNR Binary (0.20 Threshold) — Sensitivity' in 'mbcsToSbcs': - substituted for — (U+2014)”


agg_record_9db76753531d 
                      2


━━━ C-statistic comparison across LNR categorisations ━━━
               Subgroup  N Events C_statistic     SE
1              LNR_tier 62     34      0.4793 0.0460
2         LNR_tier_alt1 62     34      0.4638 0.0468
3       LNR_tier_binary 62     34      0.4839 0.0450
4 LNR_prop (continuous) 62     34      0.4667 0.0526
Saved: sensitivity_LNR_cutpoints_comparison .csv

  SUMMARY TABLE: ALL SUBGROUP & SENSITIVITY ANALYSES

Complete subgroup & sensitivity results:
                Analysis                   Subgroup  N Events C_statistic
1  TLE adequacy subgroup N-stage | Inadequate (<12) 16     13      0.4958
2  TLE adequacy subgroup     LNR | Inadequate (<12) 16     13      0.3644
3  TLE adequacy subgroup  N-stage | Adequate (>=12) 46     21      0.3875
4  TLE adequacy subgroup      LNR | Adequate (>=12) 46     21      0.4412
5       T-stage subgroup            N-stage | T3/T4 53     30      0.4230
6       T-stage subgroup                LNR | T3/T4 53     30      0.4470
7       N-sta

## 8

In [ ]:
################################################################################
#  08_css_analysis.R — Cancer-Specific Survival (Secondary Outcome)
#  Protocol Section 5.4.2 | Competing Risks (Fine-Gray)
#
#  CSS is conditional on >= 85% cause-of-death data availability.
#  Uses Fine-Gray subdistribution hazard model.
#  Non-cancer death is the competing event.
################################################################################

load(file.path(DIR_DATA, "analytic_dataset.RData"))

section_header("SCRIPT 08: CANCER-SPECIFIC SURVIVAL ANALYSIS")

cat("Analytic cohort: N =", nrow(df), "\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CSS DATA COMPLETENESS CHECK
# ═══════════════════════════════════════════════════════════════════════════════

section_header("CSS DATA COMPLETENESS CHECK")

n_deaths_os <- sum(df$Outcome_OS == 1, na.rm = TRUE)
n_deaths_dss <- sum(df$Outcome_DSS == 1, na.rm = TRUE)

# For CSS data completeness: Among deceased patients (OS=1),
# we need to know if death was cancer-specific (DSS=1) or not (DSS=0)
# All deceased patients have DSS classified since both are recorded
# Completeness = proportion of OS=1 with known cause (which is 100% here since
# DSS is recorded for all)

# Deaths attributable to cancer vs other causes
cat("Total deaths (OS events):", n_deaths_os, "\n")
cat("Cancer-specific deaths (DSS events):", n_deaths_dss, "\n")
cat("Non-cancer deaths:", n_deaths_os - n_deaths_dss, "\n")
cat("Alive/censored:", sum(df$Outcome_OS == 0), "\n\n")

# CSS completeness: all deceased have cause attribution recorded
css_completeness <- 1.0  # Both OS and DSS are recorded for everyone
cat("Cause-of-death data completeness:",
    round(css_completeness * 100, 1), "%\n")

if (css_completeness >= CSS_COD_THRESHOLD) {
  cat(">>> CSS analysis PERMITTED (>= 85% threshold MET) <<<\n\n")
  CSS_PERMITTED <- TRUE
} else {
  cat(">>> CSS analysis NOT feasible (< 85% threshold). Descriptive only. <<<\n")
  CSS_PERMITTED <- FALSE
}

# ═══════════════════════════════════════════════════════════════════════════════
# 2. COMPETING RISKS SETUP
# ═══════════════════════════════════════════════════════════════════════════════

section_header("COMPETING RISKS FRAMEWORK")

# Create competing risks event status:
#   0 = alive/censored
#   1 = cancer-specific death (event of interest)
#   2 = non-cancer death (competing event)
df$cr_status <- 0
df$cr_status[df$Outcome_DSS == 1] <- 1        # Cancer death
df$cr_status[df$Outcome_OS == 1 & df$Outcome_DSS == 0] <- 2  # Non-cancer death

cat("Competing risks event distribution:\n")
print(table(df$cr_status, dnn = "Status"))
cat("  0 = Alive/censored, 1 = Cancer death, 2 = Non-cancer death\n\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. CUMULATIVE INCIDENCE FUNCTIONS (CIF)
# ═══════════════════════════════════════════════════════════════════════════════

if (CSS_PERMITTED && n_deaths_dss >= 5) {
  section_header("CUMULATIVE INCIDENCE FUNCTIONS")

  # --- CIF by N-Stage ---
  cat("━━━ CIF by N-Stage ━━━\n")
  cif_nstage <- cmprsk::cuminc(ftime = df$Time_survival,
                                fstatus = df$cr_status,
                                group = df$N_stage)
  print(cif_nstage$Tests)

  # Plot CIF
  pdf(file.path(DIR_FIGURES, "Fig_CIF_CSS_by_Nstage.pdf"),
      width = FIG_WIDTH, height = FIG_HEIGHT)
  plot(cif_nstage, curvlab = c("N1: Cancer death", "N1: Other death",
                                 "N2: Cancer death", "N2: Other death"),
       col = c(COL_NSTAGE["N1"], adjustcolor(COL_NSTAGE["N1"], 0.4),
               COL_NSTAGE["N2"], adjustcolor(COL_NSTAGE["N2"], 0.4)),
       lty = c(1, 2, 1, 2), lwd = 2,
       main = "Cumulative Incidence: Cancer-Specific Death by N-Stage",
       xlab = "Time (days)", ylab = "Cumulative Incidence")
  legend("topleft", bty = "n",
         legend = c("N1 — Cancer", "N1 — Other", "N2 — Cancer", "N2 — Other"),
         col = c(COL_NSTAGE["N1"], adjustcolor(COL_NSTAGE["N1"], 0.4),
                 COL_NSTAGE["N2"], adjustcolor(COL_NSTAGE["N2"], 0.4)),
         lty = c(1, 2, 1, 2), lwd = 2)
  dev.off()
  cat("Saved: Fig_CIF_CSS_by_Nstage\n")

  # --- CIF by LNR Tier ---
  cat("\n━━━ CIF by LNR Tier ━━━\n")
  cif_lnr <- cmprsk::cuminc(ftime = df$Time_survival,
                              fstatus = df$cr_status,
                              group = df$LNR_tier)
  print(cif_lnr$Tests)

  pdf(file.path(DIR_FIGURES, "Fig_CIF_CSS_by_LNR_tier.pdf"),
      width = FIG_WIDTH, height = FIG_HEIGHT)
  plot(cif_lnr,
       main = "Cumulative Incidence: Cancer-Specific Death by LNR Tier",
       xlab = "Time (days)", ylab = "Cumulative Incidence",
       lwd = 2)
  dev.off()
  cat("Saved: Fig_CIF_CSS_by_LNR_tier\n")

  # ═══════════════════════════════════════════════════════════════════════════
  # 4. FINE-GRAY SUBDISTRIBUTION HAZARD MODELS
  # ═══════════════════════════════════════════════════════════════════════════

  section_header("FINE-GRAY SUBDISTRIBUTION HAZARD MODELS")

  # Prepare covariate matrix for crr()
  # crr() requires a numeric covariate matrix

  # --- Model: N-stage (Fine-Gray) ---
  cat("━━━ Fine-Gray: N-stage for CSS ━━━\n")
  cov_nstage <- model.matrix(~ N_stage, data = df)[, -1, drop = FALSE]
  fg_nstage <- cmprsk::crr(ftime = df$Time_survival,
                             fstatus = df$cr_status,
                             cov1 = cov_nstage,
                             failcode = 1, cencode = 0)
  cat("Coefficients:\n")
  print(summary(fg_nstage))

  # --- Model: LNR continuous (Fine-Gray) ---
  cat("\n━━━ Fine-Gray: LNR continuous for CSS ━━━\n")
  cov_lnr <- matrix(df$LNR_prop, ncol = 1)
  colnames(cov_lnr) <- "LNR_prop"
  fg_lnr <- cmprsk::crr(ftime = df$Time_survival,
                          fstatus = df$cr_status,
                          cov1 = cov_lnr,
                          failcode = 1, cencode = 0)
  cat("Coefficients:\n")
  print(summary(fg_lnr))

  # --- Model: LNR tiers (Fine-Gray) ---
  cat("\n━━━ Fine-Gray: LNR tiers for CSS ━━━\n")
  cov_lnr_tier <- model.matrix(~ LNR_tier, data = df)[, -1, drop = FALSE]
  fg_lnr_tier <- cmprsk::crr(ftime = df$Time_survival,
                               fstatus = df$cr_status,
                               cov1 = cov_lnr_tier,
                               failcode = 1, cencode = 0)
  cat("Coefficients:\n")
  print(summary(fg_lnr_tier))

  # ═══════════════════════════════════════════════════════════════════════════
  # 5. FULL MVA FINE-GRAY (if tier permits)
  # ═══════════════════════════════════════════════════════════════════════════

  if (ANALYTIC_TIER$permit_mva) {
    section_header("FINE-GRAY: FULL MVA FOR CSS")

    # N-stage MVA
    cat("━━━ Fine-Gray: N-stage Full MVA for CSS ━━━\n")
    cov_C_css <- model.matrix(~ N_stage + LN_total + AG + Sex_f +
                                T_stage_group + Grade_f + LVI_f,
                              data = df)[, -1]
    fg_C <- cmprsk::crr(ftime = df$Time_survival, fstatus = df$cr_status,
                         cov1 = cov_C_css, failcode = 1, cencode = 0)
    print(summary(fg_C))

    # LNR MVA
    cat("\n━━━ Fine-Gray: LNR Full MVA for CSS ━━━\n")
    cov_D_css <- model.matrix(~ LNR_prop + LN_total + AG + Sex_f +
                                T_stage_group + Grade_f + LVI_f,
                              data = df)[, -1]
    fg_D <- cmprsk::crr(ftime = df$Time_survival, fstatus = df$cr_status,
                         cov1 = cov_D_css, failcode = 1, cencode = 0)
    print(summary(fg_D))
  }

  # Save results
  save(cif_nstage, cif_lnr, fg_nstage, fg_lnr, fg_lnr_tier,
       file = file.path(DIR_MODELS, "phase_css_competing_risks.RData"))

} else {
  cat("\n  CSS analysis limited due to insufficient events or data.\n")
  cat("  Cancer-specific deaths:", n_deaths_dss, "\n")
  cat("  Reporting descriptive results only.\n")

  section_header("DESCRIPTIVE CSS ANALYSIS")

  # Descriptive: cause of death distribution
  cod_tab <- data.frame(
    Category = c("Alive/censored", "Cancer-specific death",
                 "Non-cancer death", "Total"),
    N = c(sum(df$cr_status == 0), sum(df$cr_status == 1),
          sum(df$cr_status == 2), nrow(df)),
    Pct = c(round(100 * sum(df$cr_status == 0) / nrow(df), 1),
            round(100 * sum(df$cr_status == 1) / nrow(df), 1),
            round(100 * sum(df$cr_status == 2) / nrow(df), 1),
            100)
  )
  print(cod_tab)
  save_table(cod_tab, "CSS_descriptive_cause_of_death")
}

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 08 COMPLETE\n")
cat(strrep("=", 78), "\n")



  SCRIPT 08: CANCER-SPECIFIC SURVIVAL ANALYSIS

Analytic cohort: N = 62 

  CSS DATA COMPLETENESS CHECK

Total deaths (OS events): 34 
Cancer-specific deaths (DSS events): 24 
Non-cancer deaths: 10 
Alive/censored: 28 

Cause-of-death data completeness: 100 %
>>> CSS analysis PERMITTED (>= 85% threshold MET) <<<


  COMPETING RISKS FRAMEWORK

Competing risks event distribution:
Status
 0  1  2 
28 24 10 
  0 = Alive/censored, 1 = Cancer death, 2 = Non-cancer death


  CUMULATIVE INCIDENCE FUNCTIONS

━━━ CIF by N-Stage ━━━
       stat         pv df
1 5.7758558 0.01624781  1
2 0.1391876 0.70909001  1


Warning message in text.default(x, y, ...):
“for 'N1 — Cancer' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in text.default(x, y, ...):
“for 'N1 — Other' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in text.default(x, y, ...):
“for 'N2 — Cancer' in 'mbcsToSbcs': - substituted for — (U+2014)”
Warning message in text.default(x, y, ...):
“for 'N2 — Other' in 'mbcsToSbcs': - substituted for — (U+2014)”


Saved: Fig_CIF_CSS_by_Nstage

━━━ CIF by LNR Tier ━━━
      stat        pv df
1 0.941436 0.6245537  2
2 1.476593 0.4779274  2
Saved: Fig_CIF_CSS_by_LNR_tier

  FINE-GRAY SUBDISTRIBUTION HAZARD MODELS

━━━ Fine-Gray: N-stage for CSS ━━━
Coefficients:
Competing Risks Regression

Call:
cmprsk::crr(ftime = df$Time_survival, fstatus = df$cr_status, 
    cov1 = cov_nstage, failcode = 1, cencode = 0)

           coef exp(coef) se(coef)    z p-value
N_stageN2 0.973      2.64    0.401 2.42   0.015

          exp(coef) exp(-coef) 2.5% 97.5%
N_stageN2      2.64      0.378  1.2  5.81

Num. cases = 62
Pseudo Log-likelihood = -89.8 
Pseudo likelihood ratio test = 5.44  on 1 df,

━━━ Fine-Gray: LNR continuous for CSS ━━━
Coefficients:
Competing Risks Regression

Call:
cmprsk::crr(ftime = df$Time_survival, fstatus = df$cr_status, 
    cov1 = cov_lnr, failcode = 1, cencode = 0)

          coef exp(coef) se(coef)   z p-value
LNR_prop 0.574      1.78    0.522 1.1    0.27

         exp(coef) exp(-coef)  2

## 9

In [ ]:
################################################################################
#  09_tables_figures.R — Publication Tables & Figures Compilation
#  Assembles all outputs into final publication-ready format.
#
#  Run AFTER all preceding scripts (01–08) have completed successfully.
################################################################################

load(file.path(DIR_DATA, "analytic_dataset.RData"))

# Load all model results (wrapped in tryCatch for robustness)
tryCatch(load(file.path(DIR_MODELS, "phase2_unadjusted_results.RData")),
         error = function(e) cat("Phase 2 results not found.\n"))
tryCatch(load(file.path(DIR_MODELS, "phase3_cox_models.RData")),
         error = function(e) cat("Phase 3 model results not found.\n"))
tryCatch(load(file.path(DIR_MODELS, "phase3_performance_metrics.RData")),
         error = function(e) cat("Phase 3 performance results not found.\n"))
tryCatch(load(file.path(DIR_MODELS, "phase4_reclassification.RData")),
         error = function(e) cat("Phase 4 results not found.\n"))
tryCatch(load(file.path(DIR_MODELS, "phase5_subgroup_sensitivity.RData")),
         error = function(e) cat("Phase 5 results not found.\n"))
tryCatch(load(file.path(DIR_MODELS, "phase_css_competing_risks.RData")),
         error = function(e) cat("CSS results not found.\n"))

section_header("SCRIPT 09: TABLES & FIGURES COMPILATION")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. MANUSCRIPT FIGURE 1: CONSORT FLOW DIAGRAM (Text-based)
# ═══════════════════════════════════════════════════════════════════════════════

section_header("FIGURE 1: CONSORT FLOW")

flow_text <- paste0(
  "CONSORT-Style Patient Flow Diagram\n",
  "===================================\n\n",
  "Total records in database: ", flow$step1_total, "\n",
  "  └─ Excluded (invalid LN data): ", flow$excluded_invalid_ln, "\n",
  "  └─ Excluded (missing survival): ", flow$excluded_missing_surv, "\n",
  "  └─ Excluded (node-negative): ", flow$excluded_node_negative, "\n",
  "     (Filter: ", NODE_POSITIVE_FILTER, ")\n",
  "  └─ Excluded (TLE missing/zero): ", flow$excluded_tle_missing, "\n",
  "  ────────────────────\n",
  "  FINAL ANALYTIC COHORT: ", flow$step5_final_analytic, "\n",
  "  OS events: ", sum(df$Outcome_OS == 1), "\n",
  "  DSS events: ", sum(df$Outcome_DSS == 1), "\n",
  "  Analytic tier: ", ANALYTIC_TIER$label, "\n"
)
cat(flow_text)
writeLines(flow_text, file.path(DIR_TABLES, "Figure1_CONSORT_flow.txt"))

# ═══════════════════════════════════════════════════════════════════════════════
# 2. MANUSCRIPT TABLE 2: MULTIVARIABLE COX MODEL RESULTS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("TABLE 2: MULTIVARIABLE COX MODEL RESULTS")

if (exists("models") && !is.null(models$C) && !is.null(models$D)) {
  # Model C (N-stage full MVA)
  cox_C_tidy <- coxph(Surv(Time_survival, Outcome_OS) ~
                        N_stage + LN_total + AG + Sex_f +
                        T_stage_group + Grade_f + LVI_f, data = df)
  tidy_C <- broom::tidy(cox_C_tidy, exponentiate = TRUE, conf.int = TRUE)
  tidy_C$Model <- "C (N-stage)"

  # Model D (LNR full MVA)
  cox_D_tidy <- coxph(Surv(Time_survival, Outcome_OS) ~
                        LNR_prop + LN_total + AG + Sex_f +
                        T_stage_group + Grade_f + LVI_f, data = df)
  tidy_D <- broom::tidy(cox_D_tidy, exponentiate = TRUE, conf.int = TRUE)
  tidy_D$Model <- "D (LNR)"

  table2 <- rbind(tidy_C, tidy_D)
  table2$HR_CI <- sprintf("%.2f (%.2f–%.2f)",
                           table2$estimate, table2$conf.low, table2$conf.high)
  table2$p_formatted <- sapply(table2$p.value, format_pval)

  table2_export <- table2[, c("Model", "term", "HR_CI", "p_formatted")]
  names(table2_export) <- c("Model", "Variable", "HR (95% CI)", "p-value")

  cat("Table 2: Full MVA Cox Model Results\n")
  print(table2_export)
  save_table(table2_export, "Table2_MVA_results")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 3. MANUSCRIPT TABLE 3: C-STATISTIC COMPARISON ACROSS LAYERS
# ═══════════════════════════════════════════════════════════════════════════════

section_header("TABLE 3: C-STATISTIC COMPARISON")

if (exists("c_results") && nrow(c_results) > 0) {
  table3 <- c_results[, c("Model", "C_statistic", "CI_lower", "CI_upper")]
  table3$C_CI <- sprintf("%.4f (%.4f–%.4f)",
                          table3$C_statistic, table3$CI_lower, table3$CI_upper)

  # Add layer info
  table3$Layer <- ifelse(table3$Model %in% c("A", "B"), "1 (Unadjusted)",
                  ifelse(table3$Model %in% c("C2", "D2"), "2 (TLE-adjusted)",
                         "3 (Full MVA)"))

  # Add ΔC within each layer
  table3$Delta_C <- NA
  pairs <- list(c("A", "B"), c("C2", "D2"), c("C", "D"))
  for (pair in pairs) {
    if (all(pair %in% table3$Model)) {
      c1 <- table3$C_statistic[table3$Model == pair[1]]
      c2 <- table3$C_statistic[table3$Model == pair[2]]
      table3$Delta_C[table3$Model == pair[2]] <- round(c2 - c1, 4)
    }
  }

  table3_export <- table3[, c("Layer", "Model", "C_CI", "Delta_C")]
  names(table3_export) <- c("Layer", "Model", "C-statistic (95% CI)",
                             "ΔC (LNR - N-stage)")

  cat("Table 3: C-Statistic Comparison\n")
  print(table3_export)
  save_table(table3_export, "Table3_C_statistic_comparison")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 4. MANUSCRIPT TABLE 4: RECLASSIFICATION TABLE
# ═══════════════════════════════════════════════════════════════════════════════

section_header("TABLE 4: RECLASSIFICATION")

if (exists("cross_tab")) {
  cat("N-Stage × LNR Tier Cross-Tabulation:\n")
  print(addmargins(cross_tab))

  reclass_output <- as.data.frame.matrix(addmargins(cross_tab))
  save_table(reclass_output, "Table4_reclassification")
}

# ═══════════════════════════════════════════════════════════════════════════════
# 5. MULTI-PANEL COMPOSITE FIGURES
# ═══════════════════════════════════════════════════════════════════════════════

section_header("COMPOSITE FIGURES")

# Try to create a composite KM figure (N-stage + LNR tier side by side)
tryCatch({
  km_ns <- survfit(Surv(Time_survival, Outcome_OS) ~ N_stage, data = df)
  km_lnr <- survfit(Surv(Time_survival, Outcome_OS) ~ LNR_tier, data = df)

  p1 <- ggsurvplot(km_ns, data = df, palette = unname(COL_NSTAGE),
                    pval = TRUE, conf.int = TRUE, conf.int.alpha = 0.1,
                    xlab = "Time (days)", ylab = "OS Probability",
                    title = "A. Overall Survival by N-Stage",
                    legend.title = "N-Stage", legend.labs = c("N1", "N2"),
                    ggtheme = theme_publication(base_size = 10),
                    break.time.by = 365.25, legend = "bottom")

  p2 <- ggsurvplot(km_lnr, data = df, palette = unname(COL_LNR_TIERS),
                    pval = TRUE, conf.int = TRUE, conf.int.alpha = 0.1,
                    xlab = "Time (days)", ylab = "OS Probability",
                    title = "B. Overall Survival by LNR Tier",
                    legend.title = "LNR Tier",
                    ggtheme = theme_publication(base_size = 10),
                    break.time.by = 365.25, legend = "bottom")

  # Combine using cowplot or direct arrangement
  pdf(file.path(DIR_FIGURES, "Fig_composite_KM_OS.pdf"),
      width = 14, height = 7)
  par(mfrow = c(1, 2))
  # Since ggsurvplot returns a list, we print them separately
  gridExtra::grid.arrange(p1$plot, p2$plot, ncol = 2)
  dev.off()

  cat("Saved: Fig_composite_KM_OS (combined N-stage + LNR tier)\n")
}, error = function(e) {
  cat("Composite figure generation failed:", e$message, "\n")
})

# ═══════════════════════════════════════════════════════════════════════════════
# 6. COMPREHENSIVE RESULTS SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

section_header("COMPREHENSIVE RESULTS SUMMARY")

cat("═══════════════════════════════════════════════════════════\n")
cat("  LNR vs N-Stage Prognostic Comparison — Key Findings\n")
cat("═══════════════════════════════════════════════════════════\n\n")

cat("Study Population:\n")
cat("  Analytic cohort: N =", nrow(df), "node-positive CRC patients\n")
cat("  Node-positive filter:", NODE_POSITIVE_FILTER, "\n")
cat("  OS events:", sum(df$Outcome_OS == 1), "\n")
cat("  DSS events:", sum(df$Outcome_DSS == 1), "\n")
cat("  Analytic tier:", ANALYTIC_TIER$label, "\n\n")

cat("TLE Characterisation (South Asian contextual data):\n")
cat("  Median TLE:", median(df$LN_total, na.rm = TRUE), "nodes\n")
cat("  TLE < 12 (AJCC minimum):",
    round(100 * sum(df$LN_total < 12) / nrow(df), 1), "%\n")
cat("  TLE < 8 (stringent):",
    round(100 * sum(df$LN_total < 8) / nrow(df), 1), "%\n\n")

cat("LNR Distribution:\n")
cat("  Median LNR:", round(median(df$LNR_prop, na.rm = TRUE), 3), "\n")
cat("  LNR tier distribution:\n")
print(table(df$LNR_tier))

if (exists("c_results") && nrow(c_results) > 0) {
  cat("\nC-Statistic Summary:\n")
  for (i in seq_len(nrow(c_results))) {
    cat(sprintf("  Model %s: C = %.4f\n", c_results$Model[i],
                c_results$C_statistic[i]))
  }
}

cat("\n═══════════════════════════════════════════════════════════\n")

# ═══════════════════════════════════════════════════════════════════════════════
# 7. FILE INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

section_header("OUTPUT FILE INVENTORY")

cat("Tables:\n")
table_files <- list.files(DIR_TABLES, full.names = FALSE)
for (f in table_files) cat("  ", f, "\n")

cat("\nFigures:\n")
fig_files <- list.files(DIR_FIGURES, full.names = FALSE)
for (f in fig_files) cat("  ", f, "\n")

cat("\nModels:\n")
model_files <- list.files(DIR_MODELS, full.names = FALSE)
for (f in model_files) cat("  ", f, "\n")

cat("\nAppendix:\n")
app_files <- list.files(DIR_APPENDIX, full.names = FALSE)
for (f in app_files) cat("  ", f, "\n")

cat("\n", strrep("=", 78), "\n")
cat("  SCRIPT 09 COMPLETE — ALL OUTPUTS COMPILED\n")
cat("  Total tables:", length(table_files), "\n")
cat("  Total figures:", length(fig_files), "\n")
cat(strrep("=", 78), "\n")



  SCRIPT 09: TABLES & FIGURES COMPILATION


  FIGURE 1: CONSORT FLOW

CONSORT-Style Patient Flow Diagram

Total records in database: 137
  └─ Excluded (invalid LN data): 0
  └─ Excluded (missing survival): 0
  └─ Excluded (node-negative): 75
     (Filter: LN_positive)
  └─ Excluded (TLE missing/zero): 0
  ────────────────────
  FINAL ANALYTIC COHORT: 62
  OS events: 34
  DSS events: 24
  Analytic tier: Inadequate (< 80 events)

  TABLE 2: MULTIVARIABLE COX MODEL RESULTS


  TABLE 3: C-STATISTIC COMPARISON

Table 3: C-Statistic Comparison
           Layer Model   C-statistic (95% CI) ΔC (LNR - N-stage)
1 1 (Unadjusted)     A 0.4248 (0.3418–0.5078)                 NA
2 1 (Unadjusted)     B 0.4667 (0.3637–0.5698)             0.0419
Saved: Table3_C_statistic_comparison .csv

  TABLE 4: RECLASSIFICATION

N-Stage × LNR Tier Cross-Tabulation:
     
      Low (<0.05) Intermediate (0.05-0.20) High (>0.20) Sum
  N1            7                       27            7  41
  N2            0        